# preprocessing deep aquifers five years

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
type_of_aquifer='deep'

# calculated data

In [ ]:
import pandas as pd

In [ ]:
# Define the file path
file_path = f"{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/WellDepths_deep_aquifers_CSV.csv"

# Read the CSV file
deep_aquifers_calculated_df = pd.read_csv(file_path)

# Preview the first few rows
print(deep_aquifers_calculated_df.head())

In [ ]:
deep_aquifers_calculated_df

In [ ]:
for column in deep_aquifers_calculated_df.columns:
  print(column)

In [ ]:
deep_aquifers_calculated_df['2015-07-01'][10]

In [ ]:
# Filter the row by microwatershed_id
value = deep_aquifers_calculated_df.loc[
    deep_aquifers_calculated_df['microwatershed_id'] == '00080000000000006861',
    '2016-10-01'
]

# Print the value
print("Value for 2016-10-01:", value.values[0] if not value.empty else "ID not found")

In [ ]:
# Filter the row by microwatershed_id
value = deep_aquifers_calculated_df.loc[
    deep_aquifers_calculated_df['microwatershed_id'] == '0008000000000000a20a',
    '2016-10-01'
]

# Print the value
print("Value for 2016-10-01:", value.values[0] if not value.empty else "ID not found")

In [ ]:
cols_to_check = ["2020-07-01", "2020-10-01"]

for col in cols_to_check:
    if col in deep_aquifers_calculated_df.columns:
        print(f"\n--- {col} ---")
        print(deep_aquifers_calculated_df[col].apply(type).value_counts())
        print(deep_aquifers_calculated_df[col].head())

In [ ]:
import pandas as pd
import ast
import json

In [ ]:
# Step 1: Copy the original DataFrame
deep_aquifers_renamed_df = deep_aquifers_calculated_df.copy()

# Step 2: Define metadata columns that should be skipped
metadata_cols = ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo']

# Step 3: Find columns with JSON data
json_columns = [col for col in deep_aquifers_renamed_df.columns if col not in metadata_cols]

# Step 4: Rename "WellDepth" to "ChangeinWellDepth_Calculated" in JSON cells
def rename_welldepth_key(cell):
    try:
        data = json.loads(cell)
        if 'WellDepth' in data:
            data['CWD_Calculated'] = data.pop('WellDepth')
        return json.dumps(data)
    except:
        return cell  # If not JSON, return as-is

for col in json_columns:
    deep_aquifers_renamed_df[col] = deep_aquifers_renamed_df[col].apply(rename_welldepth_key)

# Step 5: Rename column names from '2001-01' to '2001_Jan_March' and so on
month_to_season = {
    '01': 'Jan_March',
    '04': 'April_June',
    '07': 'Jul_Sept',
    '10': 'Oct_Dec'
}

rename_dict = {}
for col in deep_aquifers_renamed_df.columns:
    if col[:4].isdigit() and col[4] == '-' and col[5:7] in month_to_season:
        year = col[:4]
        month = col[5:7]
        season = month_to_season[month]
        rename_dict[col] = f"{year}_{season}"

deep_aquifers_renamed_df.rename(columns=rename_dict, inplace=True)

# Step 6: Print a few renamed columns as a sample
print("✅ Renamed columns (sample):")
print([col for col in deep_aquifers_renamed_df.columns if '_' in col and col.count('_') == 1][:10])

In [ ]:
deep_aquifers_renamed_df

In [ ]:
for column in deep_aquifers_renamed_df.columns:
  print(column)

In [ ]:
import pandas as pd
import json
import re

In [ ]:
# Step 1: Create a copy
df = deep_aquifers_renamed_df.copy()

# Step 2: Identify all seasonal columns
pattern = r"(\d{4})_(Jan_March|April_June|Jul_Sept|Oct_Dec)"
seasonal_cols = [col for col in df.columns if re.match(pattern, col)]

# Step 3: Map season name to number
season_order = {
    "Jan_March": 1,
    "April_June": 2,
    "Jul_Sept": 3,
    "Oct_Dec": 4
}

# Step 4: Melt to long format
id_vars = ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo']
df_melted = df.melt(id_vars=id_vars, value_vars=seasonal_cols,
                    var_name='year_season', value_name='json_data')

# Step 5: Extract year and season from column name
df_melted[['year', 'season_name']] = df_melted['year_season'].str.extract(r'(\d{4})_(Jan_March|April_June|Jul_Sept|Oct_Dec)')
df_melted['season_number'] = df_melted['season_name'].map(season_order).astype(int)
df_melted['year'] = df_melted['year'].astype(int)

# Step 6: Parse JSON safely
def parse_json(val):
    try:
        if isinstance(val, str):
            return json.loads(val)
        elif isinstance(val, dict):
            return val
    except:
        return {}
    return {}

parsed_df = df_melted['json_data'].apply(parse_json).apply(pd.Series)

# Step 7: Combine all together
deep_aquifer_unrolled = pd.concat([df_melted[id_vars + ['year', 'season_name', 'season_number']], parsed_df], axis=1)

In [ ]:
deep_aquifer_unrolled

In [ ]:
# List of variables to aggregate
agg_vars = ['DeltaG', 'ET', 'G', 'Precipitation', 'RunOff', 'CWD_Calculated']

# Keep only necessary columns
cols_to_keep = ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo', 'year', 'season_number'] + agg_vars
df = deep_aquifer_unrolled[cols_to_keep].copy()

# Define a mapping from original season_number to new labels
def map_season(row):
    if row['season_number'] in [1, 2]:
        return 'pre_monsoon', 1
    elif row['season_number'] in [3, 4]:
        return 'post_monsoon', 2
    else:
        return np.nan, np.nan

df[['season_name', 'new_season_number']] = df.apply(map_season, axis=1, result_type='expand')

# Drop rows where season_name is NaN (if any)
df = df.dropna(subset=['season_name'])

# Group and sum
grouped = df.groupby(
    ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo', 'year', 'season_name', 'new_season_number'],
    as_index=False
)[agg_vars].sum()

# Rename for clarity
grouped.rename(columns={'new_season_number': 'season_number'}, inplace=True)

# Final DataFrame
deep_aquifer_pre_post = grouped

In [ ]:
deep_aquifer_pre_post[deep_aquifer_pre_post['microwatershed_id']=='0008000000000000130d']

In [ ]:
deep_aquifer_pre_post

In [ ]:
import pandas as pd

In [ ]:
# Copy original data
df = deep_aquifer_pre_post.copy()
df['year'] = df['year'].astype(int)

# Covariates to average
covariates = ['DeltaG', 'ET', 'G', 'Precipitation', 'RunOff', 'CWD_Calculated']
max_year = df['year'].max()


# Container for results
lagged_data = []

for lag in range(0, 6):
    for mws_id in df['microwatershed_id'].unique():
        mws_data = df[df['microwatershed_id'] == mws_id]

        for start_year in mws_data['year'].unique():
            end_year = start_year + lag
            if end_year <= max_year:
                target_years = list(range(start_year, end_year + 1))

                subset = mws_data[mws_data['year'].isin(target_years)].copy()

                # Sort subset by year and season order
                subset = subset.sort_values(by=['year', 'season_number'])

                if (subset['year'].nunique() == (lag + 1)) and subset[covariates].notna().all().all():
                    mean_vals = subset[covariates].mean(numeric_only=True)

                    row_info = subset.iloc[0][['microwatershed_id', 'weighted_avg_yeild', '.geo', 'year']]
                    combined = pd.concat([row_info, mean_vals])
                    combined['lag'] = lag
                    lagged_data.append(combined)

# Final DataFrame
lagged_covariate_df_strict = pd.DataFrame(lagged_data).reset_index(drop=True)

In [ ]:
lagged_covariate_df_strict

In [ ]:
# Check for columns with any NaNs
missing_values = lagged_covariate_df_strict.isna().sum()

# Display only those columns that contain missing values
print(missing_values[missing_values > 0])

In [ ]:
lagged_covariate_df_strict

In [ ]:
deep_aquifer_pre_post[deep_aquifer_pre_post['microwatershed_id']=='0008000000000000130d']

In [ ]:
lagged_covariate_df_strict[
    (lagged_covariate_df_strict['microwatershed_id'] == '0008000000000000130d') &
    (lagged_covariate_df_strict['year'] == 2015)]

In [ ]:
for column in lagged_covariate_df_strict.columns:
  print(column)

*take only 5 year lag*

In [ ]:
lag_5_df = lagged_covariate_df_strict[lagged_covariate_df_strict['lag'] == 5]

In [ ]:
lag_5_df

#Terrain and Stream Order

In [ ]:
file_path = f'{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/deep_aquifer_terrain.csv'
# Read CSV into a DataFrame
terrain_filt_df = pd.read_csv(file_path)

In [ ]:
terrain_filt_df

In [ ]:
for column in terrain_filt_df.columns:
  print(column)

In [ ]:
# Merge terrain_cluster from terrain_filt_df into melted_cwd_unrolled
cwd_with_terrain_df = lag_5_df.merge(
    terrain_filt_df[['microwatershed_id', 'terrain_cluster']],
    how='left',
    on='microwatershed_id'
)

In [ ]:
cwd_with_terrain_df

In [ ]:
for column in cwd_with_terrain_df:
  print(column)

In [ ]:
cwd_with_terrain_df = cwd_with_terrain_df.dropna(subset=['terrain_cluster']).reset_index(drop=True)

In [ ]:
cwd_with_terrain_df

In [ ]:
# Count NaN values in terrain_cluster
nan_count = cwd_with_terrain_df['terrain_cluster'].isna().sum()
print("Number of NaNs in terrain_cluster:", nan_count)

# Optional: Boolean check
print("No NaNs present? ->", nan_count == 0)

*stream order*

In [ ]:
import pandas as pd

# Define the file path for reading the CSV
csv_file_path = f'{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/deep_aquifer_SO.csv'

# Read the CSV file into a DataFrame
stream_order_continous = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame to ensure it was read correctly
print(stream_order_continous)

In [ ]:
for column in stream_order_continous.columns:
  print(column)

In [ ]:
stream_order_continous.rename(
    columns={f"percent_{i}": f"Area%_SO_{i}" for i in range(1, 12)},
    inplace=True
)

In [ ]:
stream_order_continous

In [ ]:
# Define stream order columns to be added
stream_order_columns = [
    'Area%_SO_1', 'Area%_SO_2', 'Area%_SO_3', 'Area%_SO_4', 'Area%_SO_5', 'Area%_SO_6',
    'Area%_SO_7', 'Area%_SO_8', 'Area%_SO_9', 'Area%_SO_10', 'Area%_SO_11'
]

# Perform the merge using 'microwater' as key from stream_order_continous
cwd_with_terrain_and_stream_order_df = cwd_with_terrain_df.merge(
    stream_order_continous[['microwater'] + stream_order_columns],
    how='left',
    left_on='microwatershed_id',
    right_on='microwater'
).drop(columns=['microwater'])

In [ ]:
cwd_with_terrain_and_stream_order_df

In [ ]:
for column in cwd_with_terrain_and_stream_order_df.columns:
  print(column)

In [ ]:
# List of columns to check
stream_order_check_columns = [
    'Area%_SO_1', 'Area%_SO_2', 'Area%_SO_3', 'Area%_SO_4', 'Area%_SO_5',
    'Area%_SO_6', 'Area%_SO_7', 'Area%_SO_8', 'Area%_SO_9', 'Area%_SO_10', 'Area%_SO_11'
]

# Check for NaNs and print count per column
nan_counts = cwd_with_terrain_and_stream_order_df[stream_order_check_columns].isna().sum()
print("NaN counts per column:\n", nan_counts)

# Optional: Check if all columns are fully non-null
all_non_null = (nan_counts == 0).all()
print("\nAll columns have no NaNs? ->", all_non_null)

In [ ]:
lag_5_df

In [ ]:
for column in lag_5_df.columns:
  print(column)

In [ ]:
cwd_with_terrain_and_stream_order_df

adding soge

In [ ]:
import pandas as pd

In [ ]:
# Step 1: Load soge_deep
soge_deep = pd.read_csv(f'{WORKSPACE_ROOT}/20. SOGE new data/output/soge_deep.csv')

In [ ]:
soge_deep

In [ ]:
soge_deep.rename(
    columns={col: col.replace("sgw_dev_pe_", "soge_") for col in soge_deep.columns if col.startswith("sgw_dev_pe_")},
    inplace=True
)

In [ ]:
soge_deep

In [ ]:
# Melt soge_deep into long format
soge_deep_melted = soge_deep.melt(
    id_vars='microwatershed_id',
    var_name='year',
    value_name='soge'
)

# Extract the numeric year from column names like 'soge_2013'
soge_deep_melted['year'] = soge_deep_melted['year'].str.extract(r'(\d{4})').astype(int)

In [ ]:
soge_deep_melted

In [ ]:
cwd_with_terrain_and_stream_order_df

In [ ]:
# Step 1: Rename 'soge' to 'soge_t_minus_1' in the melted DataFrame
soge_deep_melted_renamed = soge_deep_melted.rename(columns={'soge': 'soge_t_minus_1'})

# Step 2: Add 1 to 'year' so it aligns with year t in the main dataframe (originally it was t-1)
soge_deep_melted_renamed['year'] = pd.to_numeric(
    soge_deep_melted_renamed['year'], errors='coerce'
) + 1

In [ ]:
soge_deep_melted_renamed

In [ ]:
# Step 3: Merge on microwatershed_id and year
final_dataframe = cwd_with_terrain_and_stream_order_df.merge(
    soge_deep_melted_renamed,
    how='left',
    on=['microwatershed_id', 'year'],
)

In [ ]:
final_dataframe

In [ ]:
nan_count = final_dataframe['soge_t_minus_1'].isna().sum()
print(f"Number of rows with NaN in soge_t_minus_1: {nan_count}")

In [ ]:
final_dataframe = final_dataframe.dropna(subset=['soge_t_minus_1'])

In [ ]:
final_dataframe

*add surface water presence*

In [ ]:
import pandas as pd

# Path to your CSV
csv_path = f'{WORKSPACE_ROOT}/19. preprocessing_deep_aquifers/input/CropSeason_AllYears_Combined_deep.csv'

# Read the CSV
surface_water_presence_deep = pd.read_csv(csv_path)

# Display the first few rows
print(surface_water_presence_deep)

In [ ]:
surface_water_presence_deep.columns

In [ ]:
for column in surface_water_presence_deep.columns:
  print(column)

In [ ]:
columns_to_keep = [
    'krkrz_by_total_pct_2017', 'krkrz_by_total_pct_2018', 'krkrz_by_total_pct_2019',
    'krkrz_by_total_pct_2020', 'krkrz_by_total_pct_2021', 'krkrz_by_total_pct_2022',
    'krz_by_total_pct_2017', 'krz_by_total_pct_2018', 'krz_by_total_pct_2019',
    'krz_by_total_pct_2020', 'krz_by_total_pct_2021', 'krz_by_total_pct_2022',
    'microwatershed_id', '.geo'
]

surface_water_presence_deep = surface_water_presence_deep[columns_to_keep]

In [ ]:
surface_water_presence_deep

In [ ]:
for column in surface_water_presence_deep.columns:
  print(column)

In [ ]:
# Melt krkrz_by_total_pct columns
krkrz_melted = surface_water_presence_deep.melt(
    id_vars='microwatershed_id',
    value_vars=[f'krkrz_by_total_pct_{year}' for year in range(2017, 2023)],
    var_name='year',
    value_name='krkrz_by_total_pct'
)
krkrz_melted['year'] = krkrz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Melt krz_by_total_pct columns
krz_melted = surface_water_presence_deep.melt(
    id_vars='microwatershed_id',
    value_vars=[f'krz_by_total_pct_{year}' for year in range(2017, 2023)],
    var_name='year',
    value_name='krz_by_total_pct'
)
krz_melted['year'] = krz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Merge both melted DataFrames on microwatershed_id and year
surface_water_presence_deep_melted = pd.merge(
    krkrz_melted,
    krz_melted,
    on=['microwatershed_id', 'year'],
    how='inner'
)

In [ ]:
surface_water_presence_deep_melted

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe = final_dataframe.merge(
    surface_water_presence_deep_melted,
    on=['microwatershed_id', 'year'],
    how='left'
)

In [ ]:
final_dataframe

In [ ]:
final_dataframe = final_dataframe.dropna(subset=['krkrz_by_total_pct', 'krz_by_total_pct']).reset_index(drop=True)

In [ ]:
final_dataframe

In [ ]:
for column in final_dataframe.columns:
  print(column)

*mws new to old mapping*

In [ ]:
file_path = f'{WORKSPACE_ROOT}/22. Training_deep_five_years/input/deep_mws_mapping.csv'

# Read CSV
new_to_old_id_mapping = pd.read_csv(file_path)

In [ ]:
new_to_old_id_mapping

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_id_mapping = final_dataframe.merge(
    new_to_old_id_mapping[['microwatershed_id', 'uid']],
    on='microwatershed_id',
    how='left'
)

In [ ]:
final_dataframe_after_id_mapping

*adding max stream order*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/max_stream_order_deep.csv"

# Read CSV
max_stream_order = pd.read_csv(file_path)

In [ ]:
max_stream_order

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_max_stream_order = final_dataframe_after_id_mapping.merge(
    max_stream_order[['microwatershed_id', 'max_stream_order']],
    on='microwatershed_id',
    how='left'
)

In [ ]:
final_dataframe_after_max_stream_order

*adding connectivity info and catchment area*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/catchment_area_connectivity_Info.csv"

# Read CSV
connectivity_catchment_area = pd.read_csv(file_path)

In [ ]:
connectivity_catchment_area

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_connectivity_catchment = final_dataframe_after_max_stream_order.merge(
    connectivity_catchment_area[['uid', 'mws_connectivity_indegree','summed_upstream_catchment_area']],
    on='uid',
    how='left'
)

In [ ]:
final_dataframe_after_connectivity_catchment

*adding cropping percentage*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/CroppingPercentage_AllYears_Combined_deep.csv"

# Read CSV
cropping_percentage = pd.read_csv(file_path)

In [ ]:
cropping_percentage

In [ ]:
for column in cropping_percentage.columns:
  print(column)

In [ ]:
# List of columns to retain
columns_to_keep = ['cropping_krkrz_by_total_pct_2017',
'cropping_krkrz_by_total_pct_2018',
'cropping_krkrz_by_total_pct_2019',
'cropping_krkrz_by_total_pct_2020',
'cropping_krkrz_by_total_pct_2021',
'cropping_krkrz_by_total_pct_2022',
'cropping_krz_by_total_pct_2017',
'cropping_krz_by_total_pct_2018',
'cropping_krz_by_total_pct_2019',
'cropping_krz_by_total_pct_2020',
'cropping_krz_by_total_pct_2021',
'cropping_krz_by_total_pct_2022',
    'microwatershed_id', '.geo'# Optional: keep for merging later
]

# Subset the DataFrame
cropping_percentage = cropping_percentage[columns_to_keep]

In [ ]:
# Step 1: Melt krkrz_by_total_pct columns
cropping_krkrz_melted = cropping_percentage.melt(
    id_vars='microwatershed_id',
    value_vars=[f'cropping_krkrz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='cropping_krkrz_by_total_pct'
)

cropping_krkrz_melted['year'] = cropping_krkrz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 2: Melt krz_by_total_pct columns
cropping_krz_melted = cropping_percentage.melt(
    id_vars='microwatershed_id',
    value_vars=[f'cropping_krz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='cropping_krz_by_total_pct'
)

cropping_krz_melted['year'] = cropping_krz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 3: Merge both melted DataFrames on microwatershed_id and year
cropping_percentage_melted = pd.merge(
    cropping_krkrz_melted,
    cropping_krz_melted,
    on=['microwatershed_id', 'year'],
    how='inner'
)

In [ ]:
cropping_percentage_melted

In [ ]:
# Merge surface water metrics into final_dataframe
final_dataframe_after_cropping_percentage = final_dataframe_after_connectivity_catchment.merge(
    cropping_percentage_melted,
    how='left',
    on=['microwatershed_id', 'year']
)

In [ ]:
final_dataframe_after_cropping_percentage

In [ ]:
final_dataframe_after_cropping_percentage = final_dataframe_after_cropping_percentage.dropna()

In [ ]:
final_dataframe_after_cropping_percentage

*infiltration rate*

In [ ]:
file_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/infiltration_combined.csv"
infiltration_rate = pd.read_csv(file_path)

In [ ]:
infiltration_rate

In [ ]:
# Drop rows with duplicate 'Code' values (keep first occurrence only)
infiltration_rate_cleaned = infiltration_rate.drop_duplicates(subset='Code', keep='first')

# Rename the 'Recommended Rate' column to 'infiltration_rate'
infiltration_rate_cleaned = infiltration_rate_cleaned.rename(columns={'Recommended Rate': 'infiltration_rate'})
infiltration_rate_cleaned = infiltration_rate_cleaned.rename(columns={'Code': 'newcode43'})

# Display the cleaned DataFrame
print(infiltration_rate_cleaned.head())

In [ ]:
infiltration_rate_cleaned

# get the filtered list of mws along with their aquifer category

In [ ]:
import pandas as pd

# Path to your pickle file
file_path = f"{WORKSPACE_ROOT}/13. Checking ABY data/output/filtered_wells_based_on_nans.csv"

# Load the pickle file as a DataFrame
thresholded_mws_ids_with_aquifer_types = pd.read_csv(file_path)

# Display the first few rows
print(thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
thresholded_mws_ids_with_aquifer_types

In [ ]:
thresholded_mws_ids_with_aquifer_types['id']

In [ ]:
for column in thresholded_mws_ids_with_aquifer_types.columns:
  print(column)

In [ ]:
thresholded_mws_ids_with_aquifer_types

In [ ]:
thresholded_mws_ids_with_aquifer_types[
    ['aquifer', 'aquifer0', 'aquifers', 'area_re', 'avg_mbgl', 'm2_perday', 'm3_per_day', 'mbgl', 'newcode14', 'newcode43']
]

In [ ]:
thresholded_mws_ids_with_aquifer_types['aquifer_category'].isnull().sum()

# removing unecessary columns

In [ ]:
# List of columns to drop
columns_to_drop = [
'Sr. No.',
'State_Name_With_LGD_Code',
'District_Name_With_LGD_Code',
'Block_Name_With_LGD_Code',
'GP_Name_With_LGD_Code',
'Village',
'Site_Name',
'TYPE',
'SOURCE',
'Well_ID',
'Well_Depth (meters)',
'Aquifer',
'index_right',
'DN',
'consecutive_non_nan_pairs',
'index_aquifer',
'id',
'aquifer',
'aquifer0',
'aquifers',
'area_re',
'avg_mbgl',
'm2_perday',
'm3_per_day',
'mbgl',
'newcode14',
#'newcode43',
'objectid',
'pa_order',
'per_cm',
'shape_Area',
'shape_Leng',
'state',
'system',
'test',
'yeild__',
'zone_m',
'longitude',
'latitude'
]

# Create a new DataFrame without the unwanted columns
filtered_thresholded_mws_ids_with_aquifer_types = thresholded_mws_ids_with_aquifer_types.drop(columns=columns_to_drop, errors='ignore')  # errors='ignore' skips missing columns

# View the result
print(filtered_thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

In [ ]:
import pandas as pd
import re
import unicodedata as ud

# Work on a copy
df = filtered_thresholded_mws_ids_with_aquifer_types.copy()

# 0) Normalize column names (strip spaces, fix weird dashes/zero-width chars)
def norm_col(c: str) -> str:
    s = ud.normalize('NFKC', c)
    s = s.replace('\u00A0', ' ')                            # NBSP -> space
    s = re.sub(r'[\u200B-\u200D\uFEFF]', '', s)            # zero-width
    s = re.sub(r'[\u2010-\u2015]', '-', s)                 # any Unicode dash -> '-'
    return s.strip()

df.rename(columns={c: norm_col(c) for c in df.columns}, inplace=True)

# 1) Collect target columns (those starting with 'Pre' or 'Post') that actually exist
pre_post_cols = [c for c in df.columns if c.startswith(('Pre','Post'))]
if not pre_post_cols:
    raise ValueError("No columns starting with 'Pre' or 'Post' found after normalization.")

# 2) Coerce to numeric (strings -> NaN), then drop rows that have NaN in ANY of these cols
df[pre_post_cols] = df[pre_post_cols].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=pre_post_cols).reset_index(drop=True)

# 3) Multiply by -1 (NaNs already removed; numeric only)
df[pre_post_cols] = df[pre_post_cols].mul(-1)

# 4) (Optional) sanity checks
print("Target columns:", pre_post_cols)
print(df[pre_post_cols].head())
print("Shape after cleaning:", df.shape)

# Keep results
filtered_thresholded_mws_ids_with_aquifer_types = df


In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

In [ ]:
# Strip whitespace and normalize dashes in column names
filtered_thresholded_mws_ids_with_aquifer_types.columns = (
    filtered_thresholded_mws_ids_with_aquifer_types.columns
    .str.strip()
    .str.replace('–', '-', regex=False)  # replace en dash with hyphen
)

# Now find Pre/Post columns
pre_post_cols = [c for c in filtered_thresholded_mws_ids_with_aquifer_types.columns
                 if c.startswith(('Pre','Post'))]



print("Flipped columns:", pre_post_cols)


In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

*changing aquifer categories to alluvial and non alluvial*

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types['aquifer_category'] = filtered_thresholded_mws_ids_with_aquifer_types['aquifer_category'].apply(
    lambda x: 'alluvial' if x == 'Alluvial' else 'non_alluvial'
)

In [ ]:
# Columns of interest
pre_post_cols = [
    'Pre-monsoon_2015', 'Post-monsoon_2015',
    'Pre-monsoon_2016', 'Post-monsoon_2016',
    'Pre-monsoon_2017', 'Post-monsoon_2017',
    'Pre-monsoon_2018', 'Post-monsoon_2018',
    'Pre-monsoon_2019', 'Post-monsoon_2019',
    'Pre-monsoon_2020', 'Post-monsoon_2020',
    'Pre-monsoon_2021', 'Post-monsoon_2021',
    'Pre-monsoon_2022', 'Post-monsoon_2022'
]

# Melt dataframe to long format
melted_df = filtered_thresholded_mws_ids_with_aquifer_types.melt(
    id_vars=['microwatershed_id', 'aquifer_category', 'geometry','newcode43'],
    value_vars=pre_post_cols,
    var_name='year_season',
    value_name='WellDepth'
)

# Step 3: Extract year and season
melted_df['season'] = melted_df['year_season'].apply(lambda x: (x.split('_')[0]))
melted_df['year'] = melted_df['year_season'].apply(lambda x: int(x.split('_')[1]))

# Remove non-numeric & NaN values
melted_df['WellDepth'] = pd.to_numeric(melted_df['WellDepth'], errors='coerce')
melted_df = melted_df.dropna(subset=['WellDepth'])

In [ ]:
melted_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Plotting
plt.figure(figsize=(20, 10))
sns.boxplot(data=melted_df, x='year_season', y='WellDepth', hue='aquifer_category')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Season & Year')
plt.ylabel('Water Level')
plt.title('Boxplot of Pre & Post-monsoon Water Levels by Aquifer Category (2015-2022)')
plt.legend(title='Aquifer Category')
plt.tight_layout()
plt.show()

In [ ]:
# Drop unnecessary columns and rename as requested
melted_df = melted_df.drop(columns=['year_season'])

In [ ]:
melted_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import pandas as pd
from scipy.stats import linregress

In [ ]:
from joblib import Parallel, delayed
from scipy.stats import theilslopes
import pymannkendall as mk
from tqdm import tqdm

In [ ]:
melted_df = melted_df[
    (melted_df['microwatershed_id'].isin(final_dataframe_after_cropping_percentage['microwatershed_id'])) &
    (melted_df['year'] >= 2017)
]

In [ ]:
melted_df

In [ ]:
# Merge melted_df with infiltration_rate_cleaned on newcode43 and Code
melted_df = melted_df.merge(
    infiltration_rate_cleaned[['newcode43', 'infiltration_rate']],
    on='newcode43',
    how='left'
)


# View the result
print(melted_df.head())

In [ ]:
melted_df

In [ ]:
# Copy of original DataFrame
df = filtered_thresholded_mws_ids_with_aquifer_types.copy()

# Define the available years
years = sorted(set(melted_df['year']))

# Ensure numeric conversion for water level columns
for season in ['Pre-monsoon', 'Post-monsoon']:
    for year in years:
        col = f"{season}_{year}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# Precompute all valid year pairs (start <= end)
year_pairs = [(start, end) for start in years for end in years if end >= start]

# Initialize result list
diff_results = []

# Iterate over microwatersheds with tqdm progress bar
for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing Microwatersheds"):
    row_dict = row.to_dict()  # Convert once for faster access

    for start_year, end_year in year_pairs:
        # Fast check: skip if both values for end_year are NaN
        if pd.isna(row_dict.get(f"Pre-monsoon_{end_year}")) and pd.isna(row_dict.get(f"Post-monsoon_{end_year}")):
            continue

        # Generate full list of seasonal columns from start_year to end_year
        col_names = [
            f"{season}_{y}"
            for y in range(start_year, end_year + 1)
            for season in ['Pre-monsoon', 'Post-monsoon']
        ]

        # Collect index-value pairs with valid (non-NaN) values
        valid_pairs = [(i, row_dict[col]) for i, col in enumerate(col_names) if col in row_dict and pd.notna(row_dict[col])]
        if len(valid_pairs) < 2:
            continue  # Need at least two points

        # Unpack for regression
        x_vals, y_vals = zip(*valid_pairs)

        try:
            slope, *_ = theilslopes(y_vals, x_vals)
            mk_result = mk.original_test(y_vals)
        except Exception:
            continue  # Skip problematic cases

        # Append results
        diff_results.append({
            'microwatershed_id': row_dict.get('microwatershed_id'),
            'aquifer_category': row_dict.get('aquifer_category'),
            'start_year': start_year,
            'end_year': end_year,
            'lag': end_year - start_year,
            'sens_slope': slope,
            'mk_trend': mk_result.trend,
            'mk_p_value': mk_result.p,
            'mk_tau': mk_result.Tau,
            'num_points': len(y_vals),
            'latitude': row_dict.get('Latitude'),
            'longitude': row_dict.get('Longitude'),
            'geometry': row_dict.get('geometry')
        })

# Convert to DataFrame
diff_sens_combined_df = pd.DataFrame(diff_results)

In [ ]:
diff_sens_combined_df

In [ ]:
# Ensure both columns are numeric
diff_sens_combined_df['sens_slope'] = pd.to_numeric(diff_sens_combined_df['sens_slope'], errors='coerce')
diff_sens_combined_df['lag'] = pd.to_numeric(diff_sens_combined_df['lag'], errors='coerce')

# Create the new column
diff_sens_combined_df['magnitude_of_slope'] = diff_sens_combined_df['sens_slope'] * diff_sens_combined_df['lag']

In [ ]:
diff_sens_combined_df

In [ ]:
# Group by aquifer_category and lag, then count the rows
lag_counts = diff_sens_combined_df.groupby(['aquifer_category', 'lag']).size().reset_index(name='count')

# Display the result
print(lag_counts)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
sns.boxplot(data=diff_sens_combined_df, x='lag', y='magnitude_of_slope', hue='aquifer_category')
plt.title("Distribution of Magnitude of Slope by Lag and Aquifer Category")
plt.xlabel("Lag (years)")
plt.ylabel("Magnitude of Slope")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Filter out lag 0 and rows with null trend
mk_valid = diff_sens_combined_df[
    (diff_sens_combined_df['lag'] > 0) &
    (diff_sens_combined_df['mk_trend'].notna())
]

# Get unique lags
unique_lags = sorted(mk_valid['lag'].unique())

# Set up subplot grid
fig, axes = plt.subplots(len(unique_lags), 1, figsize=(6, len(unique_lags) * 4))

# Plot pie chart for each lag
for idx, lag in enumerate(unique_lags):
    ax = axes[idx]
    subset = mk_valid[mk_valid['lag'] == lag]
    counts = subset['mk_trend'].value_counts()
    labels = [f"{k} ({v})" for k, v in counts.items()]
    ax.pie(counts, labels=labels, autopct='%1.1f%%', startangle=140)
    ax.set_title(f"Trend Distribution for Lag {lag} years")

plt.tight_layout()
plt.show()

In [ ]:
diff_sens_combined_df.dtypes

*take only lag of 5 years*

In [ ]:
lag_5_df = diff_sens_combined_df[diff_sens_combined_df['lag'] == 5]

In [ ]:
lag_5_df

In [ ]:
from shapely import wkt

In [ ]:
# Convert 'geometry' column from string to Point geometry
lag_5_df['geometry'] = lag_5_df['geometry'].apply(wkt.loads)

# Now extract coordinates
lag_5_df['longitude'] = lag_5_df['geometry'].apply(lambda point: point.x)
lag_5_df['latitude'] = lag_5_df['geometry'].apply(lambda point: point.y)

In [ ]:
lag_5_df

In [ ]:
for column in lag_5_df.columns:
  print(column)

In [ ]:
# Merge melted_df with infiltration_rate_cleaned on newcode43 and Code
lag_5_df = lag_5_df.merge(
    melted_df[['microwatershed_id', 'infiltration_rate']],
    on='microwatershed_id',
    how='left'
)

*grace data addition*

In [ ]:
import pandas as pd
import glob

# Path where your 7 CSVs are saved
# (adjust the path to your folder)
path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/"

# Pattern to match all the GRACE CSVs
all_files = glob.glob(path + "GRACE_EWT_AllMicrowatersheds_*.csv")

# Read and concatenate
df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())

In [ ]:
import pandas as pd

# List your files in the exact order you want

files_ordered = [
    "GRACE_EWT_AllMicrowatersheds_2017_2018_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2018_2019_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2019_2020_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2020_2021_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2021_2022_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2022_2023_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2023_2024_deep.csv"
]

# Adjust path if needed
path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/"

# Read, rename, and concat
df_list = []
for f in files_ordered:
    temp = pd.read_csv(path + f)
    temp = temp.rename(columns={"mean": "ewt_cm"})
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Final shape:", df.shape)
print(df.columns)
print(df.head())

In [ ]:
df

In [ ]:
import pandas as pd

# List your files in the exact order you want

files_ordered = [
    "GRACE_EWT_AllMicrowatersheds_2017_2018_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2018_2019_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2019_2020_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2020_2021_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2021_2022_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2022_2023_deep.csv",
    "GRACE_EWT_AllMicrowatersheds_2023_2024_deep.csv"
]

# Adjust path if needed
path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/input/"



# 2) Read, rename, and tag with file_start/file_end (parsed from filename)
dfs = []
for fname in files_ordered:
    m = re.search(r"(\d{4})_(\d{4})", fname)
    assert m, f"Could not parse years from filename: {fname}"
    fstart, fend = int(m.group(1)), int(m.group(2))

    tmp = pd.read_csv(path + fname)
    tmp = tmp.rename(columns={"mean": "ewt_cm"})  # ensure consistent name
    # Make sure date is datetime (if present)
    if "date" in tmp.columns:
        tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
        tmp["year"] = tmp["date"].dt.year
        tmp["month"] = tmp["date"].dt.month
    else:
        # If no date column, you must have year/month columns already; assert they exist
        assert {"year","month"}.issubset(tmp.columns), "Need date or (year, month) columns"

    tmp["file_start"] = fstart
    tmp["file_end"] = fend
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

# 3) Build LAG=0 rows:
#    For each file block (e.g., 2017-2018), we keep only rows whose year == file_start.
lag0 = df[df["year"] == df["file_start"]].copy()
lag0["start_year"] = lag0["file_start"]
lag0["end_year"]   = lag0["file_start"]
lag0["lag"]        = 0

# Keep just the key columns you care about
keep_cols = ["microwatershed_id", "year", "month", "ewt_cm", "start_year", "end_year", "lag"]
if "date" in lag0.columns:
    keep_cols.insert(2, "date")  # optional: keep date too
lag0 = lag0[ [c for c in keep_cols if c in lag0.columns] ]

# 4) Build LAG=5 rows:
#    For each start_year S where S+5 also exists (based on file_start presence),
#    average ewt_cm for matching microwatershed_id & month between year S and year S+5.
available_starts = sorted(df["file_start"].unique().tolist())
candidate_pairs  = [(s, s+5) for s in available_starts if (s+5) in available_starts]

lag5_list = []
for s, e in candidate_pairs:
    left  = df[df["year"] == s].copy()
    right = df[df["year"] == e].copy()

    # Keep only needed columns
    cols_base = ["microwatershed_id","month","ewt_cm"]
    if "date" in df.columns:
        cols_base = ["microwatershed_id","month","ewt_cm","date"]
    left  = left[ [c for c in cols_base if c in left.columns] ]
    right = right[[c for c in cols_base if c in right.columns] ]

    # Merge on microwatershed_id + month
    on_cols = ["microwatershed_id","month"]
    merged = pd.merge(left, right, on=on_cols, suffixes=("_s","_e"), how="inner")

    # Average ewt_cm across the two years (same mw + month)
    merged["ewt_cm"] = merged[["ewt_cm_s","ewt_cm_e"]].mean(axis=1)

    # Optionally keep one of the dates (e.g., the start-year date) if it exists
    if "date_s" in merged.columns:
        merged["date"] = merged["date_s"]  # optional: pick start-year date

    # Set lag metadata
    merged["start_year"] = s
    merged["end_year"]   = e
    merged["lag"]        = 5

    # Finalize columns
    out_cols = ["microwatershed_id","month","ewt_cm","start_year","end_year","lag"]
    if "date" in merged.columns:
        out_cols.insert(1, "date")
    lag5_list.append(merged[out_cols])

lag5 = pd.concat(lag5_list, ignore_index=True) if lag5_list else pd.DataFrame(columns=lag0.columns)

# 5) Concatenate lag=0 and lag=5
df_out = pd.concat([lag0, lag5], ignore_index=True, sort=False)

# Optional: sort for readability
sort_cols = [c for c in ["microwatershed_id","start_year","lag","month"] if c in df_out.columns]
df_out = df_out.sort_values(sort_cols).reset_index(drop=True)

print(df_out.head())
print(df_out.tail())

In [ ]:
df_out

In [ ]:
import pandas as pd

# df columns expected: microwatershed_id, date, ewt_cm  (or 'mean' to be renamed)
df = df.copy()

# Make sure we have ewt_cm
if 'ewt_cm' not in df.columns and 'mean' in df.columns:
    df = df.rename(columns={'mean': 'ewt_cm'})

# Types
df['date'] = pd.to_datetime(df['date'])
df['ewt_cm'] = pd.to_numeric(df['ewt_cm'], errors='coerce')
df['year'] = df['date'].dt.year

# -------------------------
# LAG = 0  (yearly means)
# -------------------------
yearly = (
    df.groupby(['microwatershed_id','year'], as_index=False)['ewt_cm']
      .mean()
)

lag0 = yearly.copy()
lag0['start_year'] = lag0['year']
lag0['end_year']   = lag0['year']
lag0['lag']        = 0
lag0 = lag0[['microwatershed_id','ewt_cm','start_year','end_year','lag']]

# ----------------------------------------------------------
# LAG = 5 (mean over ALL months from start_year..end_year)
# Only build a window if BOTH start_year and end_year exist
# ----------------------------------------------------------
years_available = set(df['year'].unique())
start_years = sorted(y for y in years_available if (y + 5) in years_available)

lag5_list = []
for s in start_years:
    e = s + 5
    # Filter to the full inclusive window [s .. e]
    window = df[(df['year'] >= s) & (df['year'] <= e)]
    # Average all monthly values in the window per MWS
    grouped = (window.groupby('microwatershed_id', as_index=False)['ewt_cm']
                      .mean())
    grouped['start_year'] = s
    grouped['end_year']   = e
    grouped['lag']        = 5
    lag5_list.append(grouped)

lag5 = pd.concat(lag5_list, ignore_index=True) if lag5_list else pd.DataFrame(
    columns=['microwatershed_id','ewt_cm','start_year','end_year','lag']
)

# -------------------------
# Combine & tidy
# -------------------------
df_out = pd.concat([lag0, lag5], ignore_index=True, sort=False)
df_out = df_out.sort_values(['microwatershed_id','lag','start_year']).reset_index(drop=True)

# Remove 'year' if present (from earlier steps)
df_out = df_out.drop(columns=['year'], errors='ignore')

print(df_out.head(12))
print(df_out.tail(12))


In [ ]:
df_out

In [ ]:
import numpy as np
from scipy.stats import theilslopes

# helper: Sen's slope for one window (use monthly time index directly)
def sen_slope_for_window(mws_id, start_year, end_year):
    start_for_slope = max(2018, start_year)  # per your rule
    sub = df[(df['microwatershed_id'] == mws_id) &
             (df['year'] >= start_for_slope) &
             (df['year'] <= end_year)].copy()
    sub = sub.dropna(subset=['ewt_cm']).sort_values('date')
    if len(sub) < 2:
        return np.nan  # need at least 2 points

    # x = numeric month index (0,1,2,...), y = ewt_cm values
    t_idx = np.arange(len(sub))
    y = sub['ewt_cm'].astype(float).values

    slope, intercept, lo, hi = theilslopes(y, t_idx, 0.95)
    return float(slope)

# compute slope only for lag=5 rows
lag5_mask = (df_out['lag'] == 5)
df_out.loc[lag5_mask, 'grace_slope'] = df_out.loc[lag5_mask].apply(
    lambda r: sen_slope_for_window(
        r['microwatershed_id'], int(r['start_year']), int(r['end_year'])
    ),
    axis=1
)

# lag=0 rows: set NaN
df_out.loc[~lag5_mask, 'grace_slope'] = np.nan

print(df_out.head(12))
print(df_out[df_out['lag'] == 5].head(12))


In [ ]:
df_out

In [ ]:
df_out = df_out.dropna(subset=['grace_slope']).reset_index(drop=True)

In [ ]:
df_out

In [ ]:
# Make sure lag_5_df has the same key columns
keys = ['microwatershed_id', 'start_year', 'end_year', 'lag']

lag_5_df_with_grace = lag_5_df.merge(
    df_out.loc[df_out['lag']==5, keys + ['ewt_cm','grace_slope']],
    on=keys, how='left'
)

print(lag_5_df_with_grace.head())

In [ ]:
lag_5_df_with_grace = lag_5_df_with_grace.dropna(how='any').reset_index(drop=True)

print(lag_5_df_with_grace.shape)
print(lag_5_df_with_grace.head())

# training prep

In [ ]:
lag_5_df_with_grace

In [ ]:
for column in lag_5_df_with_grace.columns:
  print(column)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt

In [ ]:
# Ensure datatypes match
lag_5_df_with_grace['start_year'] = lag_5_df_with_grace['start_year'].astype(int)
lag_5_df_with_grace['lag'] = lag_5_df_with_grace['lag'].astype(int)

final_dataframe_after_cropping_percentage['year'] = final_dataframe_after_cropping_percentage['year'].astype(int)
final_dataframe_after_cropping_percentage['lag'] = final_dataframe_after_cropping_percentage['lag'].astype(int)

# Merge on microwatershed_id, year, and lag
final_df = lag_5_df_with_grace.merge(
    final_dataframe_after_cropping_percentage,
    left_on=['microwatershed_id', 'start_year', 'lag'],
    right_on=['microwatershed_id', 'year', 'lag'],
    how='inner'  # or 'left' depending on what you want
)

In [ ]:
final_df

In [ ]:
for column in final_df.columns:
  print(column)

In [ ]:
has_nans = final_df.isna().any().any()
print("Contains NaNs:", has_nans)

# season wise model

In [ ]:
# Filter by aquifer category
deep_alluvial = final_df[final_df['aquifer_category'] == 'alluvial'].copy()
deep_non_alluvial = final_df[final_df['aquifer_category'] == 'non_alluvial'].copy()

In [ ]:
deep_alluvial

*train test split based on mws id*

In [ ]:
deep_alluvial['lag'].unique()

In [ ]:
# Separate by season (assuming a 'season' column with values 1 to 4)
def split_by_lag(df):
    return {
        5: df[df['lag'] == 5].copy(),
}

alluvial_lag = split_by_lag(deep_alluvial)
non_alluvial_lag = split_by_lag(deep_non_alluvial)

In [ ]:
alluvial_lag

In [ ]:
non_alluvial_lag

In [ ]:
for column in non_alluvial_lag[5].columns:
  print(column)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Determine appropriate range based on your data
min_slope_alluvial = alluvial_lag[5]['magnitude_of_slope'].min()
max_slope_alluvial = alluvial_lag[5]['magnitude_of_slope'].max()
min_slope_non_alluvial = non_alluvial_lag[5]['magnitude_of_slope'].min()
max_slope_non_alluvial = non_alluvial_lag[5]['magnitude_of_slope'].max()

# Combined min and max for consistent binning
min_val = min(min_slope_alluvial, min_slope_non_alluvial)
max_val = max(max_slope_alluvial, max_slope_non_alluvial)

# Create bins with spacing of 0.1
bins = np.arange(min_val, max_val + 1, 1)

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)

# Alluvial
axes[0].hist(alluvial_lag[5]['magnitude_of_slope'].dropna(), bins=bins, color='skyblue', edgecolor='black')
axes[0].set_title('Alluvial: Distribution of Magnitude of Slope')
axes[0].set_xlabel('Magnitude of Slope')
axes[0].set_ylabel('Frequency')

# Non-Alluvial
axes[1].hist(non_alluvial_lag[5]['magnitude_of_slope'].dropna(), bins=bins, color='salmon', edgecolor='black')
axes[1].set_title('Non-Alluvial: Distribution of Magnitude of Slope')
axes[1].set_xlabel('Magnitude of Slope')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Define final covariate groups
target_col = 'magnitude_of_slope'
drop_cols = ['microwatershed_id', '.geo', 'lag', 'aquifer_category','sens_slope','mk_trend','mk_p_value','mk_tau','num_points','geometry'
#,
#'DeltaG','G'
]


numerical_cols = [
    'ET', 'Precipitation',
    'DeltaG','G', # added for checking
    'RunOff', 'weighted_avg_yeild',
    'soge_t_minus_1', 'krkrz_by_total_pct', 'krz_by_total_pct',
    'max_stream_order',
'mws_connectivity_indegree','infiltration_rate',
'summed_upstream_catchment_area',
'cropping_krkrz_by_total_pct',
'cropping_krz_by_total_pct','ewt_cm','grace_slope'
] + [f'Area%_SO_{i}' for i in range(1, 4)]
#categorical_cols = ['terrain_cluster']

# Container for all splits
train_test_splits = {
    'alluvial': {},
    'non_alluvial': {}
}

# # Helper function to split by microwatershed ID
# def stratified_split(df, test_frac=0.2):
#     unique_ids = df['microwatershed_id'].unique()
#     train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, random_state=42)
#     train_df = df[df['microwatershed_id'].isin(train_ids)].copy()
#     test_df = df[df['microwatershed_id'].isin(test_ids)].copy()
#     return train_df, test_df

# # Apply splitting to all seasons within each aquifer category
# for lag_number, lag_df in alluvial_lag.items():
#     train_df, test_df = stratified_split(lag_df)
#     train_test_splits['alluvial'][lag_number] = {
#         'train': train_df,
#         'test': test_df,
#         'train_rows': len(train_df),
#         'test_rows': len(test_df)
#     }

# for lag_number, lag_df in non_alluvial_lag.items():
#     train_df, test_df = stratified_split(lag_df)
#     train_test_splits['non_alluvial'][lag_number] = {
#         'train': train_df,
#         'test': test_df,
#         'train_rows': len(train_df),
#         'test_rows': len(test_df)
#     }

#binning and modelling

In [ ]:
# from sklearn.model_selection import train_test_split
# import numpy as np
# import pandas as pd

# # Define the bins and labels
# bins = [-np.inf,-8, -3, 0, 3, 5, np.inf]
# bin_labels = [0, 1, 2, 3, 4, 5]  # optional numeric labels

# def stratified_split_by_bins(df, target_col, id_col='microwatershed_id', test_frac=0.2):
#     # Step 1: Bin the target variable
#     df['bin'] = pd.cut(df[target_col], bins=bins, labels=bin_labels)

#     # Step 2: Map bin to microwatershed ID by taking the mode bin for each ID
#     id_bin_map = df.groupby(id_col)['bin'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])

#     # Step 3: Stratified split of IDs by their bin category
#     unique_ids = id_bin_map.index.to_numpy()
#     bins_for_ids = id_bin_map.to_numpy()

#     train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, stratify=bins_for_ids, random_state=42)

#     # Step 4: Return the actual dataframes
#     train_df = df[df[id_col].isin(train_ids)].drop(columns='bin')
#     test_df = df[df[id_col].isin(test_ids)].drop(columns='bin')

#     return train_df, test_df

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd



def stratified_split_by_bins(df, target_col, bins,id_col='microwatershed_id',test_frac=0.2):
    # Step 1: Bin the target variable and extract bin indices
    df['bin'] = pd.cut(df[target_col], bins=bins)
    df['bin_index'] = df['bin'].cat.codes  # Convert to numeric bin indices

    # 🔍 Print number of samples in each bin
    bin_counts = df['bin'].value_counts().sort_index()
    print("\n📊 Sample counts per bin:")
    for b, count in zip(bin_counts.index, bin_counts.values):
        print(f"  Bin {b}: {count} samples")

    # Step 2: Map bin to microwatershed ID by taking the mode bin for each ID
    id_bin_map = df.groupby(id_col)['bin_index'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])

    # Step 3: Stratified split of IDs by their bin category
    unique_ids = id_bin_map.index.to_numpy()
    bins_for_ids = id_bin_map.to_numpy()

    train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, stratify=bins_for_ids, random_state=42)

    # Step 4: Return the actual dataframes
    train_df = df[df[id_col].isin(train_ids)].drop(columns=['bin', 'bin_index'])
    test_df = df[df[id_col].isin(test_ids)].drop(columns=['bin', 'bin_index'])

    return train_df, test_df


In [ ]:
# Define your bins
alluvial_bins = [-np.inf,-7,-5,-2,-1,0,1,2,7, np.inf]
non_alluvial_bins = [-np.inf,-7,-5,-2,-1,0,1,2,7, np.inf]

In [ ]:
train_test_splits = {
    'alluvial': {},
    'non_alluvial': {}
}

for lag_number, lag_df in alluvial_lag.items():
    train_df, test_df = stratified_split_by_bins(lag_df.copy(), target_col,alluvial_bins)
    train_test_splits['alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

for lag_number, lag_df in non_alluvial_lag.items():
    train_df, test_df = stratified_split_by_bins(lag_df.copy(), target_col,non_alluvial_bins)
    train_test_splits['non_alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

In [ ]:
train_test_splits

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

def plot_boxplot_grid(df, columns, title_prefix, plots_per_row=5):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.boxplot(y=df[col], ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")  # Remove x-label
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define boxplot columns
boxplot_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - lag {lag} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - lag {lag} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_boxplot_grid(train_df, boxplot_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_boxplot_grid(test_df, boxplot_cols, title_test, plots_per_row=5)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

def plot_histogram_grid(df, columns, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define histogram columns
histogram_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, histogram_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, histogram_cols, title_test, plots_per_row=5)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

def plot_histogram_grid(df, columns, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define histogram columns
histogram_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, histogram_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, histogram_cols, title_test, plots_per_row=5)


In [ ]:
#categorical_cols_to_plot = ['terrain_cluster']  # or ['terrain_cluster'] if not one-hot encoded

def plot_histogram_grid(df, num_cols, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(num_cols)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    idx = 0
    # Plot numerical histograms
    for col in num_cols:
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[idx])
        axes[idx].set_title(col)
        axes[idx].set_xlabel("")
        idx += 1

    # Plot categorical counts

    # Turn off unused subplots
    for j in range(idx, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Now call it with:
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, numerical_cols + [target_col],  title_train)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, numerical_cols + [target_col], title_test)

In [ ]:
# Construct a summary DataFrame
summary_data = []

for aquifer, lag_years in train_test_splits.items():
    for lag_year, info in lag_years.items():
        summary_data.append({
            'Aquifer': aquifer,
            'Lag': lag_year,
            'Train Rows': info['train_rows'],
            'Test Rows': info['test_rows']
        })

# Convert to DataFrame and display
summary_df = pd.DataFrame(summary_data)
print(summary_df)

In [ ]:
lag5_data = {
    aquifer: {lag: info}
    for aquifer, lag_years in train_test_splits.items()
    for lag, info in lag_years.items()
    if lag == 5
}

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# Loop through all train/test splits
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

        train_df = data['train'].copy()
        test_df = data['test'].copy()

        # One-hot encode categorical columns
        #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
        #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

        # Align columns between train and test
        train_cols = train_df.columns
        test_df = test_df.reindex(columns=train_cols, fill_value=0)

        # Select features
        X_train = train_df[numerical_cols].copy()
        y_train = train_df[target_col]

        X_test = test_df[numerical_cols].copy()
        y_test = test_df[target_col]

        # Convert bools to int
        for df in [X_train, X_test]:
            for col in df.select_dtypes(include='bool').columns:
                df[col] = df[col].astype(int)

        # Add constant term
        X_train_ols = sm.add_constant(X_train)
        X_test_ols = sm.add_constant(X_test)

        # Align test to train
        for col in X_train_ols.columns:
            if col not in X_test_ols.columns:
                X_test_ols[col] = 0
        X_test_ols = X_test_ols[X_train_ols.columns]

        # Ensure numeric dtype
        X_train_ols = X_train_ols.astype(float)
        y_train = y_train.astype(float)
        X_test_ols = X_test_ols.astype(float)
        y_test = y_test.astype(float)
        # Fit OLS model
        model = sm.OLS(y_train, X_train_ols).fit()

        # Predict
        y_train_pred = model.predict(X_train_ols)
        y_test_pred = model.predict(X_test_ols)

        # Calculate MAPE
        train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
        test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

        # Print summary
        print(model.summary())
        print(f"Train MAPE: {train_mape:.4f}")
        print(f"Test MAPE:  {test_mape:.4f}")

        # Scatter plots
        plt.figure(figsize=(12, 5))

        # Train scatter
        plt.subplot(1, 2, 1)
        plt.scatter(y_train, y_train_pred, alpha=0.5)
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
        plt.xlabel('Actual magnitude of slope')
        plt.ylabel('Predicted magnitude of slope')
        plt.title(f'{aquifer} – {lag} – Train')

        # Test scatter
        plt.subplot(1, 2, 2)
        plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.xlabel('Actual magnitude of slope')
        plt.ylabel('Predicted magnitude of slope')
        plt.title(f'{aquifer} – {lag} – Test')

        plt.tight_layout()
        plt.show()

*Data normalization*

In [ ]:
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [ ]:
import numpy as np
import pandas as pd

def generate_inverse_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate inverse frequency-based sample weights from a continuous target variable.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges (default: [-inf, -1, 1, inf])
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y values into bins
    bin_labels = np.digitize(y, bins) - 1  # bin indices: 0, 1, 2, ...

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Compute inverse frequencies
    bin_weights = 1.0 / bin_counts
    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights

In [ ]:
def run_normalized_ols(train_test_splits, numerical_cols, categorical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align columns between train and test
            train_cols = train_df.columns
            test_df = test_df.reindex(columns=train_cols, fill_value=0)

            # Select features
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Convert bools to int
            for df in [X_train, X_test]:
                for col in df.select_dtypes(include='bool').columns:
                    df[col] = df[col].astype(int)

            # Add constant term
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns to train
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Check for NaNs or infs
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.values).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna(axis=0)
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit weighted least squares model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()

            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Scatter plots
            plt.figure(figsize=(12, 5))

            # Train scatter
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            min_val = min(y_train.min(), y_train_pred.min())
            max_val = max(y_train.max(), y_train_pred.max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='45° line')
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.axis('equal')
            plt.legend()

            # Test scatter
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            min_val = min(y_test.min(), y_test_pred.min())
            max_val = max(y_test.max(), y_test_pred.max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='45° line')
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.axis('equal')
            plt.legend()

            plt.tight_layout()
            plt.show()


In [ ]:
y_train

In [ ]:
train_test_splits

In [ ]:
alluvial_lag

In [ ]:
import numpy as np
import pandas as pd

def generate_inverse_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate inverse frequency-based sample weights from a continuous target variable.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges (default: [-inf, -1, 1, inf])
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y values into bins
    bin_labels = np.digitize(y, bins) - 1  # bin indices: 0, 1, 2, ...

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Compute inverse frequencies
    bin_weights = 1.0 / bin_counts
    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights

In [ ]:
import numpy as np

def generate_smoothed_inverse_weights(y, bins=[-np.inf, -1, 1, np.inf], alpha=0.5, eps=1e-6, normalize=True):
    """
    Generate smoothed inverse frequency weights (using exponent alpha).

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges
    - alpha: smoothing factor (0 < alpha <= 1). Smaller => gentler weighting.
    - eps: small constant to prevent division by zero
    - normalize: normalize weights to sum to len(y)

    Returns:
    - sample_weights, bin_labels, bin_counts, bin_weights
    """
    y = np.asarray(y)
    bin_labels = np.digitize(y, bins) - 1
    bin_counts = np.bincount(bin_labels)

    bin_weights = 1.0 / (np.power(bin_counts + eps, alpha))
    if normalize:
        bin_weights *= len(y) / bin_weights.sum()

    sample_weights = bin_weights[bin_labels]
    return sample_weights, bin_labels, bin_counts, bin_weights


In [ ]:
def generate_class_balanced_weights(y, bins=[-np.inf, -1, 1, np.inf], beta=0.99, normalize=True):
    """
    Class-balanced weighting based on effective number of samples (Cui et al. 2019).

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges
    - beta: smoothing factor close to 1 (e.g. 0.9, 0.99, 0.999)
    - normalize: normalize weights to sum to len(y)

    Returns:
    - sample_weights, bin_labels, bin_counts, bin_weights
    """
    y = np.asarray(y)
    bin_labels = np.digitize(y, bins) - 1
    bin_counts = np.bincount(bin_labels)

    effective_counts = (1 - np.power(beta, bin_counts)) / (1 - beta)
    bin_weights = 1.0 / effective_counts
    if normalize:
        bin_weights *= len(y) / bin_weights.sum()

    sample_weights = bin_weights[bin_labels]
    return sample_weights, bin_labels, bin_counts, bin_weights


In [ ]:
from sklearn.neighbors import KernelDensity

def generate_density_based_weights(y, bandwidth=0.5, normalize=True):
    """
    Density-based weighting using Kernel Density Estimation (KDE).
    Each sample gets weight ~ 1/p(y).

    Parameters:
    - y: array-like, target values
    - bandwidth: KDE bandwidth (controls smoothness)
    - normalize: normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights
    - density: estimated density values for each sample
    """
    y = np.asarray(y).reshape(-1, 1)

    kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth)
    kde.fit(y)
    log_density = kde.score_samples(y)
    density = np.exp(log_density)

    sample_weights = 1.0 / (density + 1e-6)
    if normalize:
        sample_weights *= len(y) / sample_weights.sum()

    return sample_weights, density

In [ ]:
import numpy as np
import pandas as pd

# Assume y_train is a NumPy array or pandas Series
y_alluvial = alluvial_lag[5][target_col].copy()
y_non_alluvial = non_alluvial_lag[5][target_col].copy()

# Step 1: Assign bins manually
# Define your bins
# alluvial_bins = [-np.inf,-7, -3,-2,-1.7,-1.4,-1,-0.75,-0.6,-0.5,-0.25,0,0.1, 0.25,0.5,0.75, 1,1.25,1.5,1.75,1.9,2,2.1,2.25,2.4,2.5,2.75,3,3.25,3.5, 4, 5, 7.5, np.inf]
# non_alluvial_bins = [-np.inf,-10, -7, -3,-2,-1,-0.5, 0,1,2, 3, 5, np.inf]

alluvial_bins = [-np.inf, -10, -5, -2.5, -1, -0.6, -0.1, -0.05, 0, 0.05, 0.1, 0.6, 1, 2.5, 5, 10, 15, np.inf]
non_alluvial_bins = [-np.inf,-10, -3,-2,-1,-0.5, 0,0.5,1,2, 5, np.inf]

alluvial_bin_labels = np.digitize(y_alluvial, alluvial_bins) - 1
non_alluvial_bin_labels = np.digitize(y_non_alluvial, non_alluvial_bins) - 1

# Step 2: Count samples per bin
alluvial_bin_counts = np.bincount(alluvial_bin_labels)
non_alluvial_bin_counts = np.bincount(non_alluvial_bin_labels)

# Step 3: Compute inverse weights
alluvial_bin_weights = 1.0 / alluvial_bin_counts
#alluvial_bin_weights *= len(y) / alluvial_bin_weights.sum()  # Optional normalization

# Step 3: Compute inverse weights
non_alluvial_bin_weights = 1.0 / non_alluvial_bin_counts
#non_alluvial_bin_weights *= len(y) / non_alluvial_bin_weights.sum()  # Optional normalization


# Step 4: Assign weight to each sample
alluvial_sample_weights = alluvial_bin_weights[alluvial_bin_labels]
non_alluvial_sample_weights = non_alluvial_bin_weights[non_alluvial_bin_labels]

In [ ]:
print("Alluvial Bin edges:", alluvial_bins)
print("Alluvial Bin counts:", alluvial_bin_counts)
print("Alluvial Bin weights:", alluvial_bin_weights)

print("Non-Alluvial Bin edges:", non_alluvial_bins)
print("Non-Alluvial Bin counts:", non_alluvial_bin_counts)
print("Non-Alluvial Bin weights:", non_alluvial_bin_weights)

In [ ]:
print("Columns in X_train before normalization:", X_train.columns.tolist())

In [ ]:
train_test_splits

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

# residuals and design matrix
resid = model.resid
exog = model.model.exog

# Breusch–Pagan
bp_test = het_breuschpagan(resid, exog)
print("Breusch-Pagan test (LM stat, p-value, f-stat, f p-value):", bp_test)

# White's Test
white_test = het_white(resid, exog)
print("White test (LM stat, p-value, f-stat, f p-value):", white_test)

In [ ]:
robust_model = sm.OLS(y_train, X_train_ols).fit(cov_type='HC3')  # HC0, HC1, HC2, HC3 available
print(robust_model.summary())

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]




            # 🚨 Drop selected features
            # cols_to_drop = ["Precipitation", "RunOff", "ewt_cm"]  # example
            # X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            # X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]



            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]



            #y_train = np.arcsinh(y_train.values)         # fit on transformed target


            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            #Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)



            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            plt.figure(figsize=(12, 5))



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)




            unique_bins = np.unique(bin_labels)

            fig, axes = plt.subplots(1, len(unique_bins), figsize=(5*len(unique_bins), 5), sharey=True)

            #Ensure axes is iterable even if there's only one bin
            if len(unique_bins) == 1:
                axes = [axes]

            for i, bin_val in enumerate(unique_bins):
                ax = axes[i]
                mask = bin_labels == bin_val

                # Get actual values in this bin
                y_bin_actual = y_train[mask]
                y_bin_pred   = y_train_pred[mask]

                if y_bin_actual.empty:
                    continue  # skip empty bins

                # Determine bin range from `bins`
                bin_min = bins[bin_val]
                bin_max = bins[bin_val + 1] if bin_val + 1 < len(bins) else y_train.max()

                # Plot
                ax.scatter(y_bin_actual, y_bin_pred, alpha=0.5)
                ax.plot([y_bin_actual.min(), y_bin_actual.max()],
                        [y_bin_actual.min(), y_bin_actual.max()],
                        'r--')

                # Set axis range to this bin's data range
                ax.set_xlim(y_bin_actual.min(), y_bin_actual.max())
                ax.set_ylim(y_bin_actual.min(), y_bin_actual.max())

                # Titles and labels
                ax.set_title(f"Bin {bin_val}: [{bin_min:.2f}, {bin_max:.2f}]")
                ax.set_xlabel("Actual")
                if i == 0:
                    ax.set_ylabel("Predicted")

                plt.tight_layout()
                plt.show()

In [ ]:
run_ols_removing_columns(train_test_splits, numerical_cols, target_col)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        cols_to_drop = ["Precipitation", "RunOff", "ewt_cm"]  # example
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns_smooth_inverse_weights(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # 🚨 Drop selected features
            # cols_to_drop = ["Precipitation", "RunOff", "ewt_cm"]  # example
            # X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            # X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_smoothed_inverse_weights(y_train,bins,0.9)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Test plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)




            unique_bins = np.unique(bin_labels)

            fig, axes = plt.subplots(1, len(unique_bins), figsize=(5*len(unique_bins), 5), sharey=True)

            # Ensure axes is iterable even if there's only one bin
            if len(unique_bins) == 1:
                axes = [axes]

            for i, bin_val in enumerate(unique_bins):
                ax = axes[i]
                mask = bin_labels == bin_val

                # Get actual values in this bin
                y_bin_actual = y_train[mask]
                y_bin_pred   = y_train_pred[mask]

                if y_bin_actual.empty:
                    continue  # skip empty bins

                # Determine bin range from `bins`
                bin_min = bins[bin_val]
                bin_max = bins[bin_val + 1] if bin_val + 1 < len(bins) else y_train.max()

                # Plot
                ax.scatter(y_bin_actual, y_bin_pred, alpha=0.5)
                ax.plot([y_bin_actual.min(), y_bin_actual.max()],
                        [y_bin_actual.min(), y_bin_actual.max()],
                        'r--')

                # Set axis range to this bin's data range
                ax.set_xlim(y_bin_actual.min(), y_bin_actual.max())
                ax.set_ylim(y_bin_actual.min(), y_bin_actual.max())

                # Titles and labels
                ax.set_title(f"Bin {bin_val}: [{bin_min:.2f}, {bin_max:.2f}]")
                ax.set_xlabel("Actual")
                if i == 0:
                    ax.set_ylabel("Predicted")

                plt.tight_layout()
                plt.show()

In [ ]:
run_ols_removing_columns_smooth_inverse_weights(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns_class_balanced_weights(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # 🚨 Drop selected features
            # cols_to_drop = ["Precipitation", "RunOff", "ewt_cm"]  # example
            # X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            # X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_class_balanced_weights(y_train,bins,0.85)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)




            unique_bins = np.unique(bin_labels)

            fig, axes = plt.subplots(1, len(unique_bins), figsize=(5*len(unique_bins), 5), sharey=True)

            # Ensure axes is iterable even if there's only one bin
            if len(unique_bins) == 1:
                axes = [axes]

            for i, bin_val in enumerate(unique_bins):
                ax = axes[i]
                mask = bin_labels == bin_val

                # Get actual values in this bin
                y_bin_actual = y_train[mask]
                y_bin_pred   = y_train_pred[mask]

                if y_bin_actual.empty:
                    continue  # skip empty bins

                # Determine bin range from `bins`
                bin_min = bins[bin_val]
                bin_max = bins[bin_val + 1] if bin_val + 1 < len(bins) else y_train.max()

                # Plot
                ax.scatter(y_bin_actual, y_bin_pred, alpha=0.5)
                ax.plot([y_bin_actual.min(), y_bin_actual.max()],
                        [y_bin_actual.min(), y_bin_actual.max()],
                        'r--')

                # Set axis range to this bin's data range
                ax.set_xlim(y_bin_actual.min(), y_bin_actual.max())
                ax.set_ylim(y_bin_actual.min(), y_bin_actual.max())

                # Titles and labels
                ax.set_title(f"Bin {bin_val}: [{bin_min:.2f}, {bin_max:.2f}]")
                ax.set_xlabel("Actual")
                if i == 0:
                    ax.set_ylabel("Predicted")

                plt.tight_layout()
                plt.show()

In [ ]:
run_ols_removing_columns_class_balanced_weights(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns_generate_density_based_weights(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # 🚨 Drop selected features
            # cols_to_drop = ["Precipitation", "RunOff", "ewt_cm"]  # example
            # X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            # X_test = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            # if aquifer == 'alluvial':
            #     bins = alluvial_bins
            # else:
            #     bins = non_alluvial_bins

            # Generate density-based weights (no bins)
            sample_weights, density = generate_density_based_weights(y_train, bandwidth=0.2)

            print(f"Density range: {density.min():.4f} to {density.max():.4f}")
            print(f"Sample weights range: {sample_weights.min():.4f} to {sample_weights.max():.4f}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            plt.figure(figsize=(8,6))
            plt.scatter(y_train, y_train_pred, c=density, cmap="viridis", alpha=0.7)
            plt.colorbar(label="Density (KDE estimate)")
            plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
            plt.xlabel("Actual")
            plt.ylabel("Predicted")
            plt.title(f"{aquifer} – {season_label} – Train (Density-weighted)")
            plt.show()



In [ ]:
run_ols_removing_columns_generate_density_based_weights(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os
from sklearn.preprocessing import MinMaxScaler

# Path to save models
model_save_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/"


def run_ols_removing_columns(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # Normalize numeric features with MinMaxScaler (scale to [0, 1])
            scaler = MinMaxScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            # # 🚨 Drop selected features
            # cols_to_drop_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 #"RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 "infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope",
            #                          ]  # example


            # # 🚨 Drop selected features
            # cols_to_drop_non_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 "RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 #"infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope"
            #                              ]  # example


            cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope",
                                     ]  # example


            cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope"
                                         ]  # example



            if aquifer == 'alluvial':
                cols_to_drop = cols_to_drop_alluvial
            else:
                cols_to_drop = cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test = X_test.drop(columns=cols_to_drop, errors="ignore")


            numerical_cols_scaled = [col for col in numerical_cols if col not in cols_to_drop]


            # Normalize numeric features
            scaler = StandardScaler()
            X_train[numerical_cols_scaled] = scaler.fit_transform(X_train[numerical_cols_scaled])
            X_test[numerical_cols_scaled] = scaler.transform(X_test[numerical_cols_scaled])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')

            #model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()


            # Save model
            model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            joblib.dump(model, os.path.join(model_save_path, model_filename))
            print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)



            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)






            plt.tight_layout()
            plt.show()

In [ ]:
run_ols_removing_columns(train_test_splits, numerical_cols, target_col)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope",
                                     ]  # example


        # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope"
                                         ]  # example

        if aquifer == 'alluvial':
          cols_to_drop = cols_to_drop_alluvial
        else:
          cols_to_drop = cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
import numpy as np
import pandas as pd

def generate_sqrt_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate sample weights using square-root inverse frequency scaling.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges for discretizing y
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y into bins
    bin_labels = np.digitize(y, bins) - 1

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Square-root inverse weighting
    bin_weights = 1.0 / np.sqrt(bin_counts)

    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights


In [ ]:
def generate_log_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate sample weights using logarithmic inverse frequency scaling.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges for discretizing y
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y into bins
    bin_labels = np.digitize(y, bins) - 1

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Logarithmic inverse weighting
    bin_weights = 1.0 / np.log1p(bin_counts)  # log(1+count)

    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights

In [ ]:
def run_ols_with_multiple_weightings(train_test_splits, numerical_cols, target_col, weight_methods=["inverse", "sqrt", "log"]):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]
            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Normalize features
            # scaler = MinMaxScaler()
            # X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            # X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            # 🚨 Drop selected features
            cols_to_drop_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"
                                            ]  # example


                # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"

                                                ]  # example



            if aquifer == 'alluvial':
                cols_to_drop = cols_to_drop_alluvial
            else:
                cols_to_drop = cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test = X_test.drop(columns=cols_to_drop, errors="ignore")


            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align
            X_test_ols = X_test_ols.reindex(columns=X_train_ols.columns, fill_value=0)

            # Pick bins
            bins = alluvial_bins if aquifer == "alluvial" else non_alluvial_bins

            # Loop through each weighting method
            for method in weight_methods:
                if method == "inverse":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train, bins)
                elif method == "sqrt":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_sqrt_sample_weights(y_train, bins)
                elif method == "log":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_log_sample_weights(y_train, bins)

                print(f"\n--- Using {method.upper()} weighting ---")
                print(f"Bin counts: {bin_counts}")
                print(f"Bin weights: {bin_weights}")

                # Fit model
                model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')

                # Predict
                y_train_pred = model.predict(X_train_ols)
                y_test_pred = model.predict(X_test_ols)

                # Evaluate
                train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

                print(model.summary())
                print(f"Train MAPE ({method}): {train_mape:.4f}")
                print(f"Test MAPE  ({method}): {test_mape:.4f}")

                # Scatter plots
                y_min = min(y_train.min(), y_test.min())
                y_max = max(y_train.max(), y_test.max())

                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.scatter(y_train, y_train_pred, alpha=0.5)
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Train – {method.upper()}")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal")

                plt.subplot(1, 2, 2)
                plt.scatter(y_test, y_test_pred, alpha=0.5, color="orange")
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Test – {method.upper()}")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal")

                plt.suptitle(f"{aquifer} – {season_label} – {method.upper()} weighting")
                plt.tight_layout()
                plt.show()


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                "infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                 "Area%_SO_2",
                                 #"Area%_SO_3"
                                        ]  # example


            # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                "infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                 "Area%_SO_2",
                                 #"Area%_SO_3"

                                            ]  # example

        if aquifer == 'alluvial':
          cols_to_drop = cols_to_drop_alluvial
        else:
          cols_to_drop = cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
run_ols_with_multiple_weightings(train_test_splits, numerical_cols, target_col)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

def plot_covariate_distributions(train_test_splits, numerical_cols, bins=30):
    """
    Plot histogram + KDE + boxplot + QQ-plot for each numerical covariate
    in the training data. Helps decide which normalization/transform to use.
    """
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== Covariate Distributions: {aquifer} – {season_label} =====")

            X_train = data['train'][numerical_cols].copy()
            X_test  = data['test'][numerical_cols].copy()

                        # 🚨 Drop selected features
            cols_to_drop_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"
                                            ]  # example


                # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"
                                    ]

            cols_to_drop = cols_to_drop_alluvial if aquifer == 'alluvial' else cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Loop over remaining numerical covariates
            for col in X_train.columns:
                fig, axes = plt.subplots(1, 3, figsize=(18, 4))

                # Histogram + KDE
                sns.histplot(X_train[col].dropna(), bins=bins, kde=True, ax=axes[0], color="skyblue")
                axes[0].set_title(f"{col} – Histogram + KDE")

                # Boxplot
                sns.boxplot(x=X_train[col].dropna(), ax=axes[1], color="orange")
                axes[1].set_title(f"{col} – Boxplot")

                # QQ-plot
                stats.probplot(X_train[col].dropna(), dist="norm", plot=axes[2])
                axes[2].set_title(f"{col} – QQ-plot")

                plt.suptitle(f"Distribution of {col} ({aquifer}, {season_label})", fontsize=14)
                plt.tight_layout()
                plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

def plot_covariate_distributions(train_test_splits, numerical_cols, bins=30):
    """
    Plot histogram + KDE + boxplot + QQ-plot for each numerical covariate
    in the training data. Helps decide which normalization/transform to use.
    """
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== Covariate Distributions: {aquifer} – {season_label} =====")

            X_train = data['train'][numerical_cols].copy()
            X_test  = data['test'][numerical_cols].copy()

                        # 🚨 Drop selected features
            cols_to_drop_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"
                                            ]  # example


                # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    #"Area%_SO_3"
                                    ]

            cols_to_drop = cols_to_drop_alluvial if aquifer == 'alluvial' else cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")

            for col in X_train.columns:
                values = X_train[col].dropna()
                skewness = stats.skew(values)

                print(f"{col}: skewness = {skewness:.3f}")

                fig, axes = plt.subplots(1, 3, figsize=(18, 4))

                # Histogram + KDE
                sns.histplot(values, bins=bins, kde=True, ax=axes[0], color="skyblue")
                axes[0].set_title(f"{col} – Histogram + KDE")

                # Boxplot
                sns.boxplot(x=values, ax=axes[1], color="orange")
                axes[1].set_title(f"{col} – Boxplot")

                # QQ-plot
                stats.probplot(values, dist="norm", plot=axes[2])
                axes[2].set_title(f"{col} – QQ-plot")

                plt.suptitle(f"Distribution of {col} (Skew = {skewness:.2f})", fontsize=14)
                plt.tight_layout()
                plt.show()

In [ ]:
plot_covariate_distributions(train_test_splits,numerical_cols)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer

def normalize_and_plot(df, normal_cols=None, right_skewed_cols=None, left_skewed_cols=None, heavy_tail_cols=None, bins=30):
    """
    Apply distribution-aware normalization and plot before/after histograms.
    df : DataFrame
    *_cols : lists of column names for each distribution type
    """

    normal_cols = normal_cols or []
    right_skewed_cols = right_skewed_cols or []
    left_skewed_cols = left_skewed_cols or []
    heavy_tail_cols = heavy_tail_cols or []

    df_norm = df.copy()

    # --- Normal (StandardScaler) ---
    scaler = StandardScaler()
    if normal_cols:
        df_norm[normal_cols] = scaler.fit_transform(df[normal_cols])

    # --- Right-skewed (log transform) ---
    for col in right_skewed_cols:
        # Add +1 shift if non-negative, else fallback to Yeo-Johnson
        if (df[col] >= 0).all():
            df_norm[col] = np.log1p(df[col])
        else:
            pt = PowerTransformer(method="yeo-johnson")
            df_norm[col] = pt.fit_transform(df[[col]])

    # --- Left-skewed (Yeo-Johnson handles negatives) ---
    for col in left_skewed_cols:
        pt = PowerTransformer(method="yeo-johnson")
        df_norm[col] = pt.fit_transform(df[[col]])

    # --- Heavy-tailed (RobustScaler) ---
    if heavy_tail_cols:
        rscaler = RobustScaler()
        df_norm[heavy_tail_cols] = rscaler.fit_transform(df[heavy_tail_cols])

    # --- Plot Before/After ---
    for col in normal_cols + right_skewed_cols + left_skewed_cols + heavy_tail_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[0], color="steelblue")
        axes[0].set_title(f"{col} – Before")

        sns.histplot(df_norm[col].dropna(), bins=bins, kde=True, ax=axes[1], color="seagreen")
        axes[1].set_title(f"{col} – After")

        plt.suptitle(f"Normalization of {col}")
        plt.tight_layout()
        plt.show()

    return df_norm


In [ ]:
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== Covariate Distributions: {aquifer} – {season_label} =====")

        X_train = data['train'][numerical_cols].copy()
        X_test  = data['test'][numerical_cols].copy()

        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                "infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                "Area%_SO_2",
                                #"Area%_SO_3"
                                        ]  # example


        # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                "infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                "Area%_SO_2",
                                #"Area%_SO_3"
                                ]

        cols_to_drop = cols_to_drop_alluvial if aquifer == 'alluvial' else cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")


        # 🚨 Drop selected features
        # cols_to_drop_alluvial = ["Precipitation", "DeltaG", "soge_t_minus_1", "krz_by_total_pct", "mws_connectivity_indegree",
        # "summed_upstream_catchment_area", "cropping_krz_by_total_pct", "Area%_SO_3"
        #                                 ]





        if aquifer == 'alluvial':
          df_norm = normalize_and_plot(
              df=X_train,
              normal_cols=["Precipitation", "DeltaG"],
              right_skewed_cols=["soge_t_minus_1", "mws_connectivity_indegree"],
              left_skewed_cols=["krz_by_total_pct", "summed_upstream_catchment_area", "cropping_krz_by_total_pct", "Area%_SO_3"],
              heavy_tail_cols=[]
          )
        else:
          df_norm = normalize_and_plot(
              df=X_train,
              normal_cols=["DeltaG"],
              right_skewed_cols=[],
              left_skewed_cols=["krz_by_total_pct","mws_connectivity_indegree","summed_upstream_catchment_area","cropping_krz_by_total_pct","soge_t_minus_1","Area%_SO_3"],
              heavy_tail_cols=["Precipitation"]
          )

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer

# ---- 1) per-distribution normalizer (fit on train, transform train+test) ----
def normalize_by_distribution(X_train, X_test, cfg):
    """
    cfg = {
      'normal':        [...],  # z-score (StandardScaler)
      'right_skewed':  [...],  # log1p if >=0 else Yeo-Johnson
      'left_skewed':   [...],  # Yeo-Johnson
      'heavy_tail':    [...]   # RobustScaler
    }
    """
    Xtr = X_train.copy()
    Xte = X_test.copy()

    def cols_in(df, cols):  # keep only columns that exist after any dropping
        return [c for c in (cols or []) if c in df.columns]

    # Right-skewed: log1p if nonnegative; else Yeo–Johnson
    for col in cols_in(Xtr, cfg.get('right_skewed')):
        if (Xtr[col].min() >= 0) and (Xte[col].min() >= 0):
            Xtr[col] = np.log1p(Xtr[col])
            Xte[col] = np.log1p(Xte[col])
        else:
            pt = PowerTransformer(method="yeo-johnson")
            Xtr[col] = pt.fit_transform(Xtr[[col]]).ravel()
            Xte[col] = pt.transform(Xte[[col]]).ravel()

    # Left-skewed: Yeo–Johnson
    for col in cols_in(Xtr, cfg.get('left_skewed')):
        pt = PowerTransformer(method="yeo-johnson")
        Xtr[col] = pt.fit_transform(Xtr[[col]]).ravel()
        Xte[col] = pt.transform(Xte[[col]]).ravel()

    # Normal-ish: StandardScaler
    norm_cols = cols_in(Xtr, cfg.get('normal'))
    if norm_cols:
        ss = StandardScaler()
        Xtr[norm_cols] = ss.fit_transform(Xtr[norm_cols])
        Xte[norm_cols] = ss.transform(Xte[norm_cols])

    # Heavy-tailed / outliers: RobustScaler
    heavy_cols = cols_in(Xtr, cfg.get('heavy_tail'))
    if heavy_cols:
        rs = RobustScaler()
        Xtr[heavy_cols] = rs.fit_transform(Xtr[heavy_cols])
        Xte[heavy_cols] = rs.transform(Xte[heavy_cols])

    return Xtr, Xte

# ---- 2) build normalized splits dict that mirrors your original structure ----
def build_normalized_splits(train_test_splits, numerical_cols, target_col, cfg_by_aquifer,
                            cols_to_drop_alluvial=None, cols_to_drop_non_alluvial=None):
    cols_to_drop_alluvial = cols_to_drop_alluvial or []
    cols_to_drop_non_alluvial = cols_to_drop_non_alluvial or []

    splits_norm = {}
    for aquifer, seasons in train_test_splits.items():
        splits_norm[aquifer] = {}
        drop_list = cols_to_drop_alluvial if aquifer == "alluvial" else cols_to_drop_non_alluvial
        cfg = cfg_by_aquifer.get(aquifer, {})  # distribution config for this aquifer

        for season_label, data in seasons.items():
            train_df = data['train'].copy()
            test_df  = data['test'].copy()

            # select features then drop per-aquifer list (if present)
            Xtr = train_df[numerical_cols].drop(columns=drop_list, errors="ignore")
            Xte = test_df[numerical_cols].drop(columns=drop_list, errors="ignore")

            # normalize per distribution (fit on train, transform both)
            Xtr_norm, Xte_norm = normalize_by_distribution(Xtr, Xte, cfg)

            # ensure same column order/coverage
            Xte_norm = Xte_norm.reindex(columns=Xtr_norm.columns, fill_value=0.0)

            # reattach target
            train_norm = pd.concat([Xtr_norm, train_df[[target_col]]], axis=1)
            test_norm  = pd.concat([Xte_norm,  test_df[[target_col]]],  axis=1)

            splits_norm[aquifer][season_label] = {
                'train': train_norm,
                'test':  test_norm
            }

    return splits_norm

# ---- 3) example config: reuse what you were plotting with normalize_and_plot ----
norm_cfg = {
    'alluvial': {
        'normal':["Precipitation", "DeltaG"],
                  'right_skewed':["soge_t_minus_1", "mws_connectivity_indegree"],
                  'left_skewed':["krz_by_total_pct", "summed_upstream_catchment_area", "cropping_krz_by_total_pct", "Area%_SO_3"],
                  'heavy_tail':[]
    },

    'non_alluvial': {
        'normal':["DeltaG"],
                  'right_skewed':[],
                  'left_skewed':["krz_by_total_pct","mws_connectivity_indegree","summed_upstream_catchment_area","cropping_krz_by_total_pct","soge_t_minus_1","Area%_SO_3"],
                  'heavy_tail':["Precipitation"]
    }
}




# 🚨 Drop selected features
cols_to_drop_alluvial = [#"Precipitation",
                        "ET",
                        #"DeltaG",
                        "G",
                        "RunOff",
                        "weighted_avg_yeild",
                        #"soge_t_minus_1",
                        "krkrz_by_total_pct",
                        #"krz_by_total_pct",
                        "max_stream_order",
                        #"mws_connectivity_indegree",
                        "infiltration_rate",
                        #"summed_upstream_catchment_area",
                        "cropping_krkrz_by_total_pct",
                        #"cropping_krz_by_total_pct",
                        "ewt_cm",
                        "grace_slope",
                        "Area%_SO_1",
                        "Area%_SO_2",
                        #"Area%_SO_3"
                                ]  # example


# 🚨 Drop selected features
cols_to_drop_non_alluvial = [#"Precipitation",
                        "ET",
                        #"DeltaG",
                        "G",
                        "RunOff",
                        "weighted_avg_yeild",
                        #"soge_t_minus_1",
                        "krkrz_by_total_pct",
                        #"krz_by_total_pct",
                        "max_stream_order",
                        #"mws_connectivity_indegree",
                        "infiltration_rate",
                        #"summed_upstream_catchment_area",
                        "cropping_krkrz_by_total_pct",
                        #"cropping_krz_by_total_pct",
                        "ewt_cm",
                        "grace_slope",
                        "Area%_SO_1",
                        "Area%_SO_2",
                        #"Area%_SO_3"
                        ]



In [ ]:
# 1) create a normalized copy of your splits
splits_norm = build_normalized_splits(
    train_test_splits=train_test_splits,
    numerical_cols=numerical_cols,     # your original feature list
    target_col=target_col,
    cfg_by_aquifer=norm_cfg,
    cols_to_drop_alluvial=cols_to_drop_alluvial,
    cols_to_drop_non_alluvial=cols_to_drop_non_alluvial
)

# 2) call your existing runner per aquifer with the **actual** normalized cols
for aquifer in splits_norm.keys():
    # pick one season to read the list of feature columns (same across seasons here)
    first_season = next(iter(splits_norm[aquifer]))
    num_cols_norm = [c for c in splits_norm[aquifer][first_season]['train'].columns if c != target_col]

    # build a mini dict for this aquifer only and pass to your OLS function
    mini_splits = {aquifer: splits_norm[aquifer]}
    run_ols_with_multiple_weightings(
        train_test_splits=mini_splits,
        numerical_cols=num_cols_norm,     # normalized feature list for this aquifer
        target_col=target_col,
        weight_methods=["inverse","sqrt","log"]
    )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress(
    X_train, y_train, X_test, y_test,
    threshold=2,
    clf_C=1.0, clf_class_weight="balanced", clf_threshold=0.5
):
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)


    # === Train classifier with tunable hyperparams ===
    clf = LogisticRegression(
        C=clf_C,
        class_weight="balanced",
        max_iter=1000
    )
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)

    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.6, c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Extract train/test
        train_df = data['train']
        test_df  = data['test']

        # Feature columns = all except target
        X_train = train_df.drop(columns=[target_col]).values
        y_train = train_df[target_col].values
        X_test  = test_df.drop(columns=[target_col]).values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=2,
            clf_C=0.1,                # smaller = stronger regularization
            clf_class_weight="balanced",
            clf_threshold=0.4         # adjust decision threshold
        )



In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-2.0,
            clf_C=0.3,                # smaller = stronger regularization
            clf_class_weight="balanced",
            clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_compare(X_train, y_train, X_test, y_test, threshold=2):
    """
    Compare ideal routing (true labels) vs classifier routing (predicted labels).
    """

    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # Define classes (based on true actual values)
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()
    reg_high = LinearRegression()

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Helper: route predictions ===
    def route_predictions(X, labels):
        preds = np.zeros(len(X))
        for i, label in enumerate(labels):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    # 1️⃣ Ideal routing (true labels)
    y_train_pred_true = route_predictions(X_train, y_class_train)
    y_test_pred_true  = route_predictions(X_test,  y_class_test)

    # 2️⃣ Classifier routing (predicted labels)
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)
    y_train_pred_clf = route_predictions(X_train, y_class_pred_train)
    y_test_pred_clf  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    print("\n--- Metrics ---")
    print("Train MAPE (ideal routing):      ", mean_absolute_percentage_error(y_train, y_train_pred_true))
    print("Test  MAPE (ideal routing):      ", mean_absolute_percentage_error(y_test,  y_test_pred_true))
    print("Train MAPE (classifier routing): ", mean_absolute_percentage_error(y_train, y_train_pred_clf))
    print("Test  MAPE (classifier routing): ", mean_absolute_percentage_error(y_test,  y_test_pred_clf))

    # === Confusion Matrix ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    ConfusionMatrixDisplay(cm, display_labels=["≤ threshold", "> threshold"]).plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter Plots ===
    def plot_scatter(y_true, y_pred, y_class, split, mode):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.6, c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"{split} Set – {mode}")
        plt.legend(); plt.gca().set_aspect("equal","box")
        plt.show()

    # Train/Test for both routing types
    plot_scatter(y_train, y_train_pred_true, y_class_train, "Train", "Ideal Routing")
    plot_scatter(y_test,  y_test_pred_true,  y_class_test,  "Test",  "Ideal Routing")
    plot_scatter(y_train, y_train_pred_clf,  y_class_train, "Train", "Classifier Routing")
    plot_scatter(y_test,  y_test_pred_clf,   y_class_test,  "Test",  "Classifier Routing")

    return clf, reg_low, reg_high, y_train_pred_true, y_test_pred_true, y_train_pred_clf, y_test_pred_clf


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_low, reg_higher, y_train_pred, y_test_pred, y_train_pred_clf, y_test_pred_clf = classify_and_regress_compare(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-1,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress(X_train, y_train, X_test, y_test, threshold=2):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set + sanity check.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)

    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    print("\nConfusion Matrix raw values:")
    print(cm)
    print(f" Sum of confusion matrix cells = {cm.sum()}, "
          f"Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-2.0,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_with_higher_prob(X_train, y_train, X_test, y_test, threshold=-2):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set + sanity check.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)

    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    #y_prob_train = clf.predict_proba(X_train)[:, 1]   # probability of > threshold
    #y_class_pred_train = (y_prob_train > 0.7).astype(int)

    #y_class_pred_test  = clf.predict(X_test)

    y_prob = clf.predict_proba(X_test)[:, 1]   # probability of > threshold
    y_class_pred_test = (y_prob > 0.7).astype(int)


    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    print("\nConfusion Matrix raw values:")
    print(cm)
    print(f" Sum of confusion matrix cells = {cm.sum()}, "
          f"Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress_with_higher_prob(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-2.0,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

def tune_classifier(X_train, y_class_train):
    """
    Use grid search with cross-validation to tune Logistic Regression hyperparams.
    """

    # Define parameter grid
    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l1", "l2"],
        "class_weight": [None, "balanced", {0:3, 1:1},{0:2, 1:1}]
    }

    # Base logistic regression
    clf = LogisticRegression(solver="liblinear", max_iter=1000)

    # GridSearchCV setup
    grid = GridSearchCV(
        clf,
        param_grid,
        cv=5,                # 5-fold cross-validation
        scoring="f1",        # focus on F1 score
        n_jobs=-1            # parallelize
    )

    grid.fit(X_train, y_class_train)

    print("Best Parameters:", grid.best_params_)
    print("Best CV F1 Score:", grid.best_score_)

    return grid.best_estimator_


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_grid_search(X_train, y_train, X_test, y_test, threshold=-2):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set + sanity check.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)

    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = tune_classifier(X_train, y_class_train)


    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    y_prob_train = clf.predict_proba(X_train)[:, 1]   # probability of > threshold
    y_class_pred_train = (y_prob_train > 0.7).astype(int)

    #y_class_pred_test  = clf.predict(X_test)

    y_prob = clf.predict_proba(X_test)[:, 1]   # probability of > threshold
    y_class_pred_test = (y_prob > 0.75).astype(int)


    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    print("\nConfusion Matrix raw values:")
    print(cm)
    print(f" Sum of confusion matrix cells = {cm.sum()}, "
          f"Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress_grid_search(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-2.0,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

# ----------------- helpers -----------------
def confusion_counts(y_true, y_pred):
    """
    Returns TP, FP, FN, TN using labels [0, 1].
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return tp, fp, fn, tn, cm

def dice_from_counts(tp, fp, fn, eps=1e-9):
    return (2.0 * tp) / (2.0 * tp + fp + fn + eps)

def jaccard_from_counts(tp, fp, fn, eps=1e-9):
    return tp / (tp + fp + fn + eps)

def tversky_from_counts(tp, fp, fn, alpha=0.5, beta=0.5, eps=1e-9):
    return tp / (tp + alpha * fp + beta * fn + eps)

# Optional soft versions if you want to score probabilities directly
def soft_dice(y_true, y_prob, eps=1e-9):
    """
    y_true in {0,1}, y_prob in [0,1]
    """
    y_true = y_true.astype(np.float64)
    y_prob = y_prob.astype(np.float64)
    intersection = np.sum(y_true * y_prob)
    return (2.0 * intersection) / (np.sum(y_true) + np.sum(y_prob) + eps)

def soft_tversky(y_true, y_prob, alpha=0.5, beta=0.5, eps=1e-9):
    y_true = y_true.astype(np.float64)
    y_prob = y_prob.astype(np.float64)
    tp = np.sum(y_true * y_prob)
    fp = np.sum((1 - y_true) * y_prob)
    fn = np.sum(y_true * (1 - y_prob))
    return tp / (tp + alpha * fp + beta * fn + eps)

# ----------------- your function with metrics -----------------
def classify_and_regress_grid_search(X_train, y_train, X_test, y_test, threshold=-2,
                                     alpha=0.5, beta=0.5,
                                     train_cls_thresh=0.70, test_cls_thresh=0.75):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set and print Dice, Jaccard, Tversky.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # Classes based on actual values
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # Train classifier
    clf = tune_classifier(X_train, y_class_train)

    # Train regressors
    reg_low  = LinearRegression()   # for y <= threshold
    reg_high = LinearRegression()   # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # Predict class labels and probabilities
    y_prob_train = clf.predict_proba(X_train)[:, 1]
    y_class_pred_train = (y_prob_train > train_cls_thresh).astype(int)

    y_prob_test = clf.predict_proba(X_test)[:, 1]
    y_class_pred_test = (y_prob_test > test_cls_thresh).astype(int)

    # Route predictions through the two regressors
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # Regression quality
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # Classification quality on train
    tp_tr, fp_tr, fn_tr, tn_tr, cm_tr = confusion_counts(y_class_train, y_class_pred_train)
    dice_tr     = dice_from_counts(tp_tr, fp_tr, fn_tr)
    jaccard_tr  = jaccard_from_counts(tp_tr, fp_tr, fn_tr)
    tversky_tr  = tversky_from_counts(tp_tr, fp_tr, fn_tr, alpha=alpha, beta=beta)

    print("\nTrain classification metrics")
    print(f"  Dice:     {dice_tr:.4f}")
    print(f"  Jaccard:  {jaccard_tr:.4f}")
    print(f"  Tversky(alpha={alpha}, beta={beta}): {tversky_tr:.4f}")

    # Optional soft versions on train
    print(f"  Soft Dice:     {soft_dice(y_class_train, y_prob_train):.4f}")
    print(f"  Soft Tversky:  {soft_tversky(y_class_train, y_prob_train, alpha=alpha, beta=beta):.4f}")

    # Classification quality on test
    tp_te, fp_te, fn_te, tn_te, cm_te = confusion_counts(y_class_test, y_class_pred_test)
    dice_te     = dice_from_counts(tp_te, fp_te, fn_te)
    jaccard_te  = jaccard_from_counts(tp_te, fp_te, fn_te)
    tversky_te  = tversky_from_counts(tp_te, fp_te, fn_te, alpha=alpha, beta=beta)

    print("\nTest classification metrics")
    print(f"  Dice:     {dice_te:.4f}")
    print(f"  Jaccard:  {jaccard_te:.4f}")
    print(f"  Tversky(alpha={alpha}, beta={beta}): {tversky_te:.4f}")

    # Optional soft versions on test
    print(f"  Soft Dice:     {soft_dice(y_class_test, y_prob_test):.4f}")
    print(f"  Soft Tversky:  {soft_tversky(y_class_test, y_prob_test, alpha=alpha, beta=beta):.4f}")

    # Confusion Matrix plot on test
    print("\nConfusion Matrix raw values (test):")
    print(cm_te)
    print(f" Sum of confusion matrix cells = {cm_te.sum()}, Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm_te,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # Scatter plots
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress_grid_search(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-2.0,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import confusion_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Tversky definition
def tversky_from_counts(tp, fp, fn, alpha=0.5, beta=0.5, eps=1e-9):
    return tp / (tp + alpha * fp + beta * fn + eps)

from sklearn.base import BaseEstimator, ClassifierMixin

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, est=None, threshold=0.5):
        self.est = est
        self.threshold = float(threshold)

    def fit(self, X, y):
        self.est.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.est.predict_proba(X)

    def predict(self, X):
        p = self.predict_proba(X)[:, 1]
        return (p > self.threshold).astype(int)

    # expose params in a namespaced way so sklearn.clone works
    def get_params(self, deep=True):
        params = {"est": self.est, "threshold": self.threshold}
        if deep and hasattr(self.est, "get_params"):
            for k, v in self.est.get_params(deep=True).items():
                params[f"est__{k}"] = v
        return params

    def set_params(self, **params):
        for k, v in params.items():
            if k == "threshold":
                self.threshold = float(v)
            elif k == "est":
                self.est = v
            elif k.startswith("est__"):
                sub_k = k.split("est__", 1)[1]
                self.est.set_params(**{sub_k: v})
            else:
                # ignore unknown keys
                pass
        return self


def oof_confusion_matrix(estimator, X, y, params, cv_splits=5, random_state=42):
    """
    Out of fold predictions only on train to build a confusion matrix
    Honest evaluation for a given parameter set
    """
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    y_pred = np.zeros_like(y, dtype=int)

    for tr, va in skf.split(X, y):
        est = clone(estimator)
        est.set_params(**params)
        est.fit(X[tr], y[tr])
        y_pred[va] = est.predict(X[va])

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    return cm

def print_cm(cm, title=None):
    tn, fp, fn, tp = cm.ravel()
    if title:
        print(title)
    print("    Confusion matrix [rows true, cols pred]")
    print(f"        [[TN={tn:4d}, FP={fp:4d}]")
    print(f"         [FN={fn:4d}, TP={tp:4d}]]")
    return tp, fp, fn, tn

def plot_cm(cm, title):
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar(fraction=0.046, pad=0.04)
    ticks = np.arange(2)
    plt.xticks(ticks, ["<= thr", "> thr"])
    plt.yticks(ticks, ["<= thr", "> thr"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

def classifier_tversky_grid_and_test(
    X_train, y_train_cont,
    X_test, y_test_cont,
    threshold_value=-2.0,
    alpha=0.5, beta=0.5,
    cv_splits=5, random_state=42,
    plot_each_matrix=False
):
    """
    Runs grid search with Tversky objective
    Prints the confusion matrix for every hyperparameter on train via out of fold predictions
    Picks the best Tversky, refits on full train, and prints the confusion matrix on test
    """

    # Build binary labels from continuous target
    y_train_bin = (np.asarray(y_train_cont) > threshold_value).astype(int)
    y_test_bin  = (np.asarray(y_test_cont)  > threshold_value).astype(int)

    # Default models and grids
    models_and_grids = [
      (
          "LogisticRegression",
          ThresholdWrapper(
              est=LogisticRegression(max_iter=2000, solver="lbfgs", random_state=random_state),
              threshold=0.5
          ),
          {
              "est__C": [0.05, 0.1, 0.5, 1.0, 5.0],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
      (
          "RandomForest",
          ThresholdWrapper(
              est=RandomForestClassifier(n_estimators=400, random_state=random_state, n_jobs=-1),
              threshold=0.5
          ),
          {
              "est__max_depth": [None, 6, 12],
              "est__min_samples_leaf": [1, 3, 5],
              "est__max_features": ["sqrt", "log2"],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
      (
          "GradientBoosting",
          ThresholdWrapper(
              est=GradientBoostingClassifier(random_state=random_state),
              threshold=0.5
          ),
          {
              "est__n_estimators": [150, 300],
              "est__learning_rate": [0.05, 0.1],
              "est__max_depth": [2, 3],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
  ]


    all_rows = []
    print(f"\nTversky objective with alpha={alpha} and beta={beta}")
    for name, base_est, grid in models_and_grids:
        print(f"\nModel family: {name}")
        for params in ParameterGrid(grid):
            cm = oof_confusion_matrix(base_est, X_train, y_train_bin, params,
                                      cv_splits=cv_splits, random_state=random_state)
            tn, fp, fn, tp = cm.ravel()
            tv = tversky_from_counts(tp, fp, fn, alpha=alpha, beta=beta)
            all_rows.append({
                "model": name,
                "params": params,
                "cm": cm,
                "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                "tversky": float(tv),
            })
            tp_, fp_, fn_, tn_ = print_cm(cm, title=f"  params={params}  Tversky={tv:.4f}")
            if plot_each_matrix:
                plt.figure(figsize=(3.8, 3.8))
                plot_cm(cm, title=f"{name}\n{params}\nTversky={tv:.3f}")
                plt.tight_layout()
                plt.show()

    # Select best by Tversky
    all_rows.sort(key=lambda r: r["tversky"], reverse=True)
    best = all_rows[0]
    print("\nBest by Tversky on out of fold train")
    print(f"  Model:  {best['model']}")
    print(f"  Params: {best['params']}")
    print(f"  Score:  {best['tversky']:.4f}")
    print_cm(best["cm"])

    # Refit best on full train and evaluate once on test
    print("\nRefit best on full train and evaluate on test")
    # Find the matching estimator blueprint
    name_to_est = {m[0]: m[1] for m in models_and_grids}
    final_est = clone(name_to_est[best["model"]])
    final_est.set_params(**best["params"])
    final_est.fit(X_train, y_train_bin)
    y_pred_test = final_est.predict(X_test)
    cm_test = confusion_matrix(y_test_bin, y_pred_test, labels=[0, 1])
    tp_t, fp_t, fn_t, tn_t = print_cm(cm_test, title="  Test confusion matrix")
    tv_test = tversky_from_counts(tp_t, fp_t, fn_t, alpha=alpha, beta=beta)
    print(f"  Test Tversky={tv_test:.4f}")

    # Plot test matrix
    plt.figure(figsize=(3.8, 3.8))
    plot_cm(cm_test, title=f"Test — {best['model']}\n{best['params']}\nTversky={tv_test:.3f}")
    plt.tight_layout()
    plt.show()

    return {
        "all_rows_sorted": all_rows,   # every hyperparameter with its OOF confusion matrix and Tversky
        "best": best,                  # best on OOF
        "cm_test": cm_test,            # test confusion matrix of the refit best model
        "tversky_test": tv_test
    }


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Split into train/test
        train_df = data["train"]
        test_df  = data["test"]

        # Feature matrix and target vector
        num_cols_norm = [c for c in train_df.columns if c != target_col]
        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        # --- call the classifier-only function ---
        result = classifier_tversky_grid_and_test(
            X_train=X_train,
            y_train_cont=y_train,
            X_test=X_test,
            y_test_cont=y_test,
            threshold_value=-2.0,  # same as before
            alpha=0.7,             # weight for false positives
            beta=0.3,              # weight for false negatives
            cv_splits=5,
            plot_each_matrix=True
        )

        # result contains everything if you want to store or log it
        best_model = result["best"]["model"]
        best_params = result["best"]["params"]
        print(f"\nBest classifier for {aquifer} – {season_label}: "
              f"{best_model} with {best_params} "
              f"(Test Tversky={result['tversky_test']:.4f})")


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Split into train/test
        train_df = data["train"]
        test_df  = data["test"]

        # Feature matrix and target vector
        num_cols_norm = [c for c in train_df.columns if c != target_col]
        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        # --- call the classifier-only function ---
        result = classifier_tversky_grid_and_test(
            X_train=X_train,
            y_train_cont=y_train,
            X_test=X_test,
            y_test_cont=y_test,
            threshold_value=-2.0,  # same as before
            alpha=0.3,             # weight for false positives
            beta=0.7,              # weight for false negatives
            cv_splits=5,
            plot_each_matrix=True
        )

        # result contains everything if you want to store or log it
        best_model = result["best"]["model"]
        best_params = result["best"]["params"]
        print(f"\nBest classifier for {aquifer} – {season_label}: "
              f"{best_model} with {best_params} "
              f"(Test Tversky={result['tversky_test']:.4f})")


In [ ]:
import itertools
import numpy as np
import pandas as pd

from statsmodels.api import OLS, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif_df(X: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame with VIF per column in X.
    Adds a constant inside and drops it from the returned table.
    """
    Xc = add_constant(X, has_constant="add")
    vif_vals = []
    cols = Xc.columns.tolist()
    for i, col in enumerate(cols):
        vif_vals.append(variance_inflation_factor(Xc.values, i))
    vif_df = pd.DataFrame({"feature": cols, "vif": vif_vals})
    vif_df = vif_df[vif_df["feature"] != "const"].reset_index(drop=True)
    return vif_df

def fit_adj_r2(X: pd.DataFrame, y: np.ndarray) -> float:
    """
    Returns adjusted R2 from statsmodels OLS on X predicting y.
    """
    Xc = add_constant(X, has_constant="add")
    model = OLS(y, Xc, hasconst=True).fit()
    return float(model.rsquared_adj)

def search_low_vif_subsets(
    df: pd.DataFrame,
    target_col: str,
    candidate_cols: list,
    threshold_value: float,
    vif_limit: float = 5.0,
    min_features: int = 4,
    verbose: bool = True
):
    """
    Exhaustive search over all combinations from full set down to min_features,
    on the subset where y <= threshold_value.

    Keeps only combos where every individual VIF < vif_limit.
    Ranks by min max VIF, then min mean VIF, then max adjusted R2.

    Returns a dict with:
      best_result: dict with keys combo, vif_df, max_vif, mean_vif, adj_r2, k, n_kept
      all_valid: list of dicts for all valid combos considered
    """
    # subset rows where y <= threshold
    sub = df[df[target_col] <= threshold_value].copy()
    y = sub[target_col].values

    p = len(candidate_cols)
    if p < min_features:
        raise ValueError(f"Need at least {min_features} candidate features, got {p}")

    all_valid = []
    if verbose:
        print(f"Total candidates: {p}. Searching sizes from {p} down to {min_features} on {len(sub)} rows")

    for k in range(p, min_features - 1, -1):
        if verbose:
            print(f"\nEvaluating combinations of size k = {k}")
        for combo in itertools.combinations(candidate_cols, k):
            X = sub[list(combo)]
            # skip degenerate columns
            if X.shape[1] != k:
                continue
            # compute VIFs
            vif_df = compute_vif_df(X)
            max_vif = float(vif_df["vif"].max())
            if max_vif >= vif_limit:
                continue  # discard this combo

            mean_vif = float(vif_df["vif"].mean())
            adj_r2 = fit_adj_r2(X, y)

            all_valid.append({
                "k": k,
                "combo": combo,
                "vif_df": vif_df,
                "max_vif": max_vif,
                "mean_vif": mean_vif,
                "adj_r2": adj_r2
            })

        if verbose:
            kept = sum(1 for r in all_valid if r["k"] == k)
            print(f"  kept {kept} combos at size {k} with all VIF < {vif_limit}")

    if not all_valid:
        if verbose:
            print("No combination satisfied the VIF constraint")
        return {"best_result": None, "all_valid": []}

    # rank
    all_valid.sort(key=lambda r: (r["max_vif"], r["mean_vif"], -r["adj_r2"]))
    best = all_valid[0]
    n_kept = sum(1 for r in all_valid if r["k"] == best["k"])
    best["n_kept"] = n_kept

    if verbose:
        print("\nBest subset by VIF ranking")
        print(f"  size k = {best['k']}")
        print(f"  combo = {best['combo']}")
        print(f"  max VIF = {best['max_vif']:.3f}, mean VIF = {best['mean_vif']:.3f}, adj R2 = {best['adj_r2']:.4f}")
        print("  VIF table:")
        print(best["vif_df"].to_string(index=False))

    return {"best_result": best, "all_valid": all_valid}


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        train_df = data["train"]
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        res = search_low_vif_subsets(
            df=train_df,
            target_col=target_col,
            candidate_cols=num_cols_norm,
            threshold_value=-2.0,  # your current split
            vif_limit=5.0,
            min_features=4,
            verbose=True
        )

        best = res["best_result"]
        if best is None:
            print("No valid subset with all VIF below 5")
            continue

        # If you want the chosen features for regression later
        chosen_features = list(best["combo"])
        print(f"Chosen features for low side regression: {chosen_features}")


In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.api import OLS, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- helpers ---
def compute_vif_df(X: pd.DataFrame) -> pd.DataFrame:
    Xc = add_constant(X, has_constant="add")
    vif_vals = []
    cols = Xc.columns.tolist()
    for i, col in enumerate(cols):
        vif_vals.append(variance_inflation_factor(Xc.values, i))
    vif_df = pd.DataFrame({"feature": cols, "vif": vif_vals})
    vif_df = vif_df[vif_df["feature"] != "const"].reset_index(drop=True)
    return vif_df

def fit_adj_r2(X: pd.DataFrame, y: np.ndarray) -> float:
    Xc = add_constant(X, has_constant="add")
    model = OLS(y, Xc, hasconst=True).fit()
    return float(model.rsquared_adj)

def search_low_vif_and_plot_regressions(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_col: str,
    candidate_cols: list,
    threshold_value: float,
    vif_limit: float = 5.0,
    min_features: int = 4,
    verbose: bool = True
):
    """
    For y <= threshold_value:
    - Tests every subset of candidate_cols from full set down to min_features
    - Keeps those with all VIF < vif_limit
    - Fits regression on train subset, plots Actual vs Predicted on train and test
    """

    # --- subset train and test where y <= threshold ---
    train_sub = train_df[train_df[target_col] <= threshold_value].copy()
    test_sub  = test_df[test_df[target_col] <= threshold_value].copy()

    y_train = train_sub[target_col].values
    y_test  = test_sub[target_col].values

    p = len(candidate_cols)
    all_valid = []
    print(f"Total covariates: {p}. Searching from {p} down to {min_features}")

    for k in range(p, min_features - 1, -1):
        print(f"\n--- Checking subsets with {k} features ---")
        for combo in itertools.combinations(candidate_cols, k):
            X_train = train_sub[list(combo)]
            X_test  = test_sub[list(combo)]
            vif_df = compute_vif_df(X_train)
            max_vif = float(vif_df["vif"].max())

            if max_vif >= vif_limit:
                continue  # skip multicollinear sets

            mean_vif = float(vif_df["vif"].mean())
            adj_r2 = fit_adj_r2(X_train, y_train)

            # Fit regression on this combo
            model = OLS(y_train, add_constant(X_train, has_constant="add")).fit()
            y_train_pred = model.predict(add_constant(X_train, has_constant="add"))
            y_test_pred  = model.predict(add_constant(X_test, has_constant="add"))

            # store info
            all_valid.append({
                "k": k,
                "combo": combo,
                "max_vif": max_vif,
                "mean_vif": mean_vif,
                "adj_r2": adj_r2,
                "vif_df": vif_df
            })

            # --- scatter plots ---
            plt.figure(figsize=(10,4))

            plt.subplot(1,2,1)
            plt.scatter(y_train, y_train_pred, color="blue", alpha=0.6, s=25)
            min_val = min(y_train.min(), y_train_pred.min())
            max_val = max(y_train.max(), y_train_pred.max())
            plt.plot([min_val, max_val],[min_val,max_val],'k--')
            plt.title(f"Train ({combo})")
            plt.xlabel("Actual"); plt.ylabel("Predicted")
            plt.gca().set_aspect("equal","box")

            plt.subplot(1,2,2)
            plt.scatter(y_test, y_test_pred, color="red", alpha=0.6, s=25)
            min_val = min(y_test.min(), y_test_pred.min())
            max_val = max(y_test.max(), y_test_pred.max())
            plt.plot([min_val, max_val],[min_val,max_val],'k--')
            plt.title(f"Test ({combo})")
            plt.xlabel("Actual"); plt.ylabel("Predicted")
            plt.gca().set_aspect("equal","box")

            plt.suptitle(
                f"Subset size {k} | max VIF={max_vif:.2f}, mean VIF={mean_vif:.2f}, adj R²={adj_r2:.3f}",
                fontsize=10
            )
            plt.tight_layout()
            plt.show()

            if verbose:
                print(f"  Combo: {combo}")
                print(f"  max VIF={max_vif:.3f}, mean VIF={mean_vif:.3f}, adj R²={adj_r2:.4f}")
                print(vif_df.to_string(index=False))
                print()

    if not all_valid:
        print("No subset satisfied the VIF criterion.")
        return None

    # sort best
    all_valid.sort(key=lambda r: (r["max_vif"], r["mean_vif"], -r["adj_r2"]))
    best = all_valid[0]
    print("\nBest subset by VIF + adjusted R²:")
    print(f"  combo={best['combo']}, max VIF={best['max_vif']:.3f}, mean VIF={best['mean_vif']:.3f}, adj R²={best['adj_r2']:.4f}")
    print(best["vif_df"].to_string(index=False))
    return all_valid


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        train_df = data["train"]
        test_df  = data["test"]

        num_cols_norm = [c for c in train_df.columns if c != target_col]

        results = search_low_vif_and_plot_regressions(
            train_df=train_df,
            test_df=test_df,
            target_col=target_col,
            candidate_cols=num_cols_norm,
            threshold_value=-2.0,   # your threshold for regression subset
            vif_limit=5.0,
            min_features=4,
            verbose=True
        )


In [ ]:
from sklearn.preprocessing import MinMaxScaler

def fit_focal_linear_regression(X, y, loss="focal_mse", gamma=2.0, lr=1e-3, epochs=500):
    """
    Train linear regression with focalized loss using gradient descent.

    Parameters
    ----------
    X : numpy array (n_samples, n_features)
    y : numpy array (n_samples,)
    loss : "focal_mse" or "focal_mae"
    gamma : focal exponent
    lr : learning rate
    epochs : iterations

    Returns
    -------
    beta : learned coefficients (n_features,)
    loss_history : list of losses over epochs
    """
    n, d = X.shape
    beta = np.zeros(d)   # initialize weights
    loss_history = []

    for epoch in range(epochs):
        y_pred = X.dot(beta)
        residual = y - y_pred

        if loss == "focal_mse":
            weights = np.abs(residual) ** gamma
            grad = -2 * (X.T.dot(weights * residual)) / n
            loss_val = focal_mse_loss(y, y_pred, gamma=gamma)

        elif loss == "focal_mae":
            weights = np.abs(residual) ** gamma
            grad = -((X.T.dot(weights * np.sign(residual)))) / n
            loss_val = focal_mae_loss(y, y_pred, gamma=gamma)

        else:
            raise ValueError("loss must be 'focal_mse' or 'focal_mae'")

        beta -= lr * grad
        loss_history.append(loss_val)

    return beta, loss_history


In [ ]:
import numpy as np

def focal_mse_loss(y_true, y_pred, gamma=2.0):
    residual = y_true - y_pred
    return np.mean((np.abs(residual) ** gamma) * (residual ** 2))

def focal_mae_loss(y_true, y_pred, gamma=1.0):
    residual = y_true - y_pred
    return np.mean(np.abs(residual) ** (1 + gamma))


In [ ]:
def run_focal_regression(train_test_splits, numerical_cols, target_col, gamma=2.0):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== Focal Regression: {aquifer} – {season_label} =====")

            # Keep them as DataFrames
            X_train = data['train'][numerical_cols].copy()
            y_train = data['train'][target_col].values
            X_test  = data['test'][numerical_cols].copy()
            y_test  = data['test'][target_col].values



            # 🚨 Drop selected features
            cols_to_drop_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    "Area%_SO_3"
                                            ]  # example


            # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                                    "ET",
                                    #"DeltaG",
                                    "G",
                                    "RunOff",
                                    "weighted_avg_yeild",
                                    #"soge_t_minus_1",
                                    "krkrz_by_total_pct",
                                    #"krz_by_total_pct",
                                    "max_stream_order",
                                    #"mws_connectivity_indegree",
                                    "infiltration_rate",
                                    #"summed_upstream_catchment_area",
                                    "cropping_krkrz_by_total_pct",
                                    #"cropping_krz_by_total_pct",
                                    "ewt_cm",
                                    "grace_slope",
                                    "Area%_SO_1",
                                    "Area%_SO_2",
                                    "Area%_SO_3"

                                                ]  # example


            cols_to_drop = cols_to_drop_alluvial if aquifer == "alluvial" else cols_to_drop_non_alluvial

            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")

            # Align test columns to train columns
            X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

            # Convert to numpy and add intercept
            X_train = np.c_[np.ones(X_train.shape[0]), X_train.values]
            X_test  = np.c_[np.ones(X_test.shape[0]), X_test.values]

            # Add constant column for intercept
            X_train = np.c_[np.ones(X_train.shape[0]), X_train]
            X_test  = np.c_[np.ones(X_test.shape[0]), X_test]

            # Fit model with focal MSE
            beta, loss_hist = fit_focal_linear_regression(X_train, y_train,
                                                         loss="focal_mse", gamma=gamma)

            y_train_pred = X_train.dot(beta)
            y_test_pred  = X_test.dot(beta)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape  = mean_absolute_percentage_error(y_test, y_test_pred)
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Plot Actual vs Predicted
            y_min, y_max = min(y_train.min(), y_test.min()), max(y_train.max(), y_test.max())
            plt.figure(figsize=(12,5))
            plt.subplot(1,2,1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min,y_max],[y_min,y_max],'r--')
            plt.title("Train – Focal Regression")
            plt.gca().set_aspect("equal","box")

            plt.subplot(1,2,2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color="orange")
            plt.plot([y_min,y_max],[y_min,y_max],'r--')
            plt.title("Test – Focal Regression")
            plt.gca().set_aspect("equal","box")
            plt.show()

In [ ]:
run_focal_regression(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np

def classify_and_regress(X_train, y_train, X_test, y_test, threshold=2):
    """
    Step 1: Train a classifier (near zero vs away from zero).
    Step 2: Train separate regressors for each group.
    """
    # Labels: 0 = near zero, 1 = away from zero
    y_class = ((y_train.abs() > threshold).astype(int))

    # Train classifier
    clf = LogisticRegression()
    clf.fit(X_train, y_class)

    # Train regressors
    reg_near = LinearRegression()
    reg_away = LinearRegression()

    reg_near.fit(X_train[y_class == 0], y_train[y_class == 0])
    reg_away.fit(X_train[y_class == 1], y_train[y_class == 1])

    # Predict
    y_class_pred = clf.predict(X_test)
    y_pred = np.zeros_like(y_test)

    # Route predictions to appropriate regressor
    for i, label in enumerate(y_class_pred):
        if label == 0:
            y_pred[i] = reg_near.predict(X_test[i].reshape(1, -1))
        else:
            y_pred[i] = reg_away.predict(X_test[i].reshape(1, -1))

    print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))
    return clf, reg_near, reg_away, y_pred


In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
model_save_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/"


def run_normalized_ols(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()


            # Save model
            model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            joblib.dump(model, os.path.join(model_save_path, model_filename))
            print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()

In [ ]:
run_normalized_ols(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def plot_covariate_scaling(train_df, numerical_cols):
    """
    Plot covariate distributions before and after normalization.
    """

    # Copy the data
    X_train_raw = train_df[numerical_cols].copy()

    # Normalize with StandardScaler
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train_raw),
        columns=numerical_cols,
        index=train_df.index
    )

    # Plot distributions
    for col in numerical_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Before normalization
        axes[0].hist(X_train_raw[col].dropna(), bins=30, color="skyblue", edgecolor="black")
        axes[0].set_title(f"Before Normalization: {col}")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Frequency")

        # After normalization
        axes[1].hist(X_train_scaled[col].dropna(), bins=30, color="salmon", edgecolor="black")
        axes[1].set_title(f"After Normalization: {col}")
        axes[1].set_xlabel(f"{col} (scaled)")
        axes[1].set_ylabel("Frequency")

        plt.tight_layout()
        plt.show()

    return X_train_scaled


In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Path to save models
#model_save_path = "[private path removed]


def run_normalized_proper_ols(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Normalize numeric features
            scaler = StandardScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            # Example: visualize normalization effect on training data
            X_train_scaled = plot_covariate_scaling(train_df, numerical_cols)


            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()

In [ ]:
run_normalized_proper_ols(train_test_splits, numerical_cols, target_col)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Assume y_train is a NumPy array or pandas Series
y = y_train.copy()

# Step 1: Assign bins manually
bins = [-np.inf,-10, -3,-0.5,0, 3,np.inf]
bin_labels = np.digitize(y, bins) - 1

# Step 2: Count samples per bin
bin_counts = np.bincount(bin_labels)

# Step 3: Compute inverse weights
bin_weights = 1.0 / bin_counts
bin_weights *= len(y) / bin_weights.sum()  # Optional normalization

# Step 4: Assign weight to each sample
sample_weights = bin_weights[bin_labels]

In [ ]:
import numpy as np
import pandas as pd

# Assume y_train is a NumPy array or pandas Series
y = y_train.copy()

# Step 1: Assign bins manually
bins = [-np.inf, -1, -0.5, 0, 1, 2, np.inf]
bin_labels = np.digitize(y, bins) - 1

# Step 2: Count samples per bin
bin_counts = np.bincount(bin_labels)

# Step 3: Compute smoothed inverse weights (log-based smoothing)
# Adding 1 inside log to avoid log(0), which is undefined
smoothed_bin_weights = 1.0 / (np.log(bin_counts + 1))

# Optional normalization so weights sum to len(y)
smoothed_bin_weights *= len(y) / smoothed_bin_weights.sum()

# Step 4: Assign weight to each sample
sample_weights = smoothed_bin_weights[bin_labels]


In [ ]:
import numpy as np
import pandas as pd

def generate_inverse_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate inverse frequency-based sample weights from a continuous target variable.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges (default: [-inf, -1, 1, inf])
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y values into bins
    bin_labels = np.digitize(y, bins) - 1  # bin indices: 0, 1, 2, ...

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Compute inverse frequencies
    bin_weights = 1.0 / bin_counts
    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights

In [ ]:
run_normalized_ols(train_test_splits, numerical_cols, target_col)

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd

def run_all_regressions_fixed(train_test_splits, numerical_cols, target_col, alpha=1.0):
    degree = 3
    model_dict = {
        'Ridge': Ridge(alpha=alpha),
        'Lasso': Lasso(alpha=alpha),
        'ElasticNet': ElasticNet(alpha=alpha, l1_ratio=0.5),
        'SVR': SVR(C=1.0),
        'k-NN': KNeighborsRegressor(n_neighbors=5),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'MLP': MLPRegressor(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),

    }

    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            if season_label != 5:
                continue  # Only run for Lag 5

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            # train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            # test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)
            # test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Features
            # X_train = train_df[numerical_cols + [col for col in train_df.columns if 'terrain_cluster_' in col]]
            # y_train = train_df[target_col]
            # X_test = test_df[numerical_cols + [col for col in test_df.columns if 'terrain_cluster_' in col]]
            # y_test = test_df[target_col]

            # # Normalize
            # if aquifer == 'alluvial':
            #     X_train[numerical_cols] = alluvial_transformer.fit_transform(X_train[numerical_cols])
            #     X_test[numerical_cols] = alluvial_transformer.transform(X_test[numerical_cols])
            # else:
            #     X_train[numerical_cols] = non_alluvial_transformer.fit_transform(X_train[numerical_cols])
            #     X_test[numerical_cols] = non_alluvial_transformer.transform(X_test[numerical_cols])

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins


            # Inverse sample weights
            sample_weights, _, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"\n===== {aquifer.upper()} – Lag 5 =====")
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            for model_name, model in model_dict.items():
                print(f"\n--- {model_name} Regression ---")

                # Fit model with sample weights if supported
                if model_name in ['Ridge', 'Lasso', 'ElasticNet']:
                    model.fit(X_train, y_train, sample_weight=sample_weights)
                else:
                    model.fit(X_train, y_train)

                # Predict
                y_train_pred = model.predict(X_train)
                y_test_pred = model.predict(X_test)

                # Metrics
                train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
                r2_train = r2_score(y_train, y_train_pred)
                r2_test = r2_score(y_test, y_test_pred)

                print(f"Train R²: {r2_train:.4f}, Test R²: {r2_test:.4f}")
                print(f"Train MAPE: {train_mape:.4f}, Test MAPE: {test_mape:.4f}")

                # Plot - separate for each model
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                for idx, (true, pred, title) in enumerate([
                    (y_train, y_train_pred, 'Train'),
                    (y_test, y_test_pred, 'Test')
                ]):
                    color = 'orange' if title == 'Test' else 'blue'
                    axes[idx].scatter(true, pred, alpha=0.5, color=color)
                    min_val, max_val = min(true.min(), pred.min()), max(true.max(), pred.max())
                    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--')
                    axes[idx].set_xlabel('Actual')
                    axes[idx].set_ylabel('Predicted')
                    axes[idx].set_title(f'{aquifer} – {season_label} – {model_name} – {title}')
                    axes[idx].axis('equal')

                plt.suptitle(f'{model_name} Regression for {aquifer} – Lag {season_label}')
                plt.tight_layout()
                plt.show()

            # === Polynomial Regression separately ===
            print(f"\n--- Polynomial Regression (degree={degree}) ---")
            poly = PolynomialFeatures(degree=degree)
            X_train_poly = poly.fit_transform(X_train)
            X_test_poly = poly.transform(X_test)

            poly_model = LinearRegression()
            poly_model.fit(X_train_poly, y_train)
            y_train_pred = poly_model.predict(X_train_poly)
            y_test_pred = poly_model.predict(X_test_poly)

            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
            r2_train = r2_score(y_train, y_train_pred)
            r2_test = r2_score(y_test, y_test_pred)

            print(f"Train R²: {r2_train:.4f}, Test R²: {r2_test:.4f}")
            print(f"Train MAPE: {train_mape:.4f}, Test MAPE: {test_mape:.4f}")

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            for idx, (true, pred, title) in enumerate([
                (y_train, y_train_pred, 'Train'),
                (y_test, y_test_pred, 'Test')
            ]):
                color = 'orange' if title == 'Test' else 'blue'
                axes[idx].scatter(true, pred, alpha=0.5, color=color)
                min_val, max_val = min(true.min(), pred.min()), max(true.max(), pred.max())
                axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--')
                axes[idx].set_xlabel('Actual')
                axes[idx].set_ylabel('Predicted')
                axes[idx].set_title(f'{aquifer} – {season_label} – Polynomial – {title}')
                axes[idx].axis('equal')

            plt.suptitle(f'Polynomial Regression (degree={degree}) for {aquifer} – Lag {season_label}')
            plt.tight_layout()
            plt.show()

In [ ]:
run_all_regressions_fixed(train_test_splits, numerical_cols, target_col, alpha=1.0)

#getting shallow outputs on deep model

In [ ]:
# Define the path
save_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/output/"

# Read shallow alluvial
shallow_alluvial = pd.read_csv(save_path + "shallow_alluvial.csv")

# Read shallow non-alluvial
shallow_non_alluvial = pd.read_csv(save_path + "shallow_non_alluvial.csv")

In [ ]:
shallow_alluvial

In [ ]:
shallow_non_alluvial

In [ ]:
covariate_cols = [col for col in train_df.columns if col not in ['microwatershed_id','year','start_year','end_year', '.geo', 'lag', 'aquifer_category','sens_slope','mk_trend','mk_p_value','mk_tau','num_points','geometry','DeltaG','G','Area%_SO_1',
 'Area%_SO_2',
 'Area%_SO_3',
 'Area%_SO_4',
 'Area%_SO_5',
 'Area%_SO_6',
 'Area%_SO_7',
 'Area%_SO_8',
 'Area%_SO_9',
 'Area%_SO_10',
 'Area%_SO_11',
 'CWD_Calculated',
  'latitude',
 'longitude',
  'uid',
  'terrain_cluster_2.0']]

In [ ]:
covariate_cols

In [ ]:
import pandas as pd
import numpy as np
import pandas.api.types as ptypes

def summarize_covariates(train_df, test_df, covariate_cols):
    summary = []

    for col in covariate_cols:
        if not ptypes.is_numeric_dtype(train_df[col]):
            continue  # Skip non-numeric columns

        train_stats = train_df[col].describe()
        test_stats = test_df[col].describe()

        summary.append({
            'Covariate': col,
            'Train Mean': train_stats['mean'],
            'Test Mean': test_stats['mean'],
            'Train Std': train_stats['std'],
            'Test Std': test_stats['std'],
            'Train Min': train_stats['min'],
            'Test Min': test_stats['min'],
            'Train Max': train_stats['max'],
            'Test Max': test_stats['max']
        })

    return pd.DataFrame(summary)

In [ ]:
model_dir = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/"

In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
        print(f"\n=== {aquifer.upper()} – LAG {lag} ===")
        train_df = train_test_splits[aquifer][lag]['train']
        if (aquifer=='alluvial'):
          test_df = shallow_alluvial
        else:
          test_df = shallow_non_alluvial
        summary_df = summarize_covariates(train_df, test_df, covariate_cols)
        display(summary_df)  # or print(summary_df.to_string(index=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_covariate_distributions(train_df, test_df, covariate_cols, title_prefix=""):
    for col in covariate_cols:
        plt.figure(figsize=(8, 4))
        sns.kdeplot(train_df[col], label='Train', shade=True)
        #sns.kdeplot(test_df[col], label='Test', shade=True)
        plt.title(f'{title_prefix} Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Density')
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_covariate_distributions_train_test(train_df, test_df, covariate_cols, title_prefix=""):
    for col in covariate_cols:
        plt.figure(figsize=(8, 4))
        sns.kdeplot(train_df[col], label='Training data of Deeper model', shade=True)
        sns.kdeplot(test_df[col], label='Testing on Shallow data', shade=True)
        plt.title(f'{title_prefix} Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Density')
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
      if (aquifer=='alluvial'):
          test_df = shallow_alluvial
      else:
          test_df = shallow_non_alluvial
      plot_covariate_distributions(train_df, test_df, covariate_cols, title_prefix=f"{aquifer.upper()} LAG {lag}")

In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
      if (aquifer=='alluvial'):
          test_df = shallow_alluvial
      else:
          test_df = shallow_non_alluvial
      plot_covariate_distributions_train_test(train_df, test_df, covariate_cols, title_prefix=f"{aquifer.upper()} LAG {lag}")

In [ ]:
from scipy.stats import wasserstein_distance
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
      if (aquifer=='alluvial'):
          test_df = shallow_alluvial
      else:
          test_df = shallow_non_alluvial
      for col in covariate_cols:
        dist = wasserstein_distance(train_df[col].dropna(), test_df[col].dropna())
        print(f'{col}: Wasserstein Distance = {dist:.4f}')


In [ ]:
covariate_cols

In [ ]:
covariate_cols = [col for col in covariate_cols if col != "magnitude_of_slope"]

In [ ]:
import shap
import os
import pandas as pd
import joblib
import statsmodels.api as sm
season_label=5
covariate_cols
for aquifer in {'alluvial','non_alluvial'}:
  if (aquifer=='alluvial'):
    test_df = shallow_alluvial
  else:
    test_df = shallow_non_alluvial
  model_path = os.path.join(model_dir, f"{aquifer}_{season_label}_ols_model.pkl")
  model = joblib.load(model_path)
  explainer = shap.Explainer(model.predict, train_df[covariate_cols])
  shap_values_train = explainer(train_df[covariate_cols])
  shap_values_new = explainer(test_df[covariate_cols])


  shap.summary_plot(shap_values_train.values, train_df[covariate_cols], show=False)
  plt.title(f"Train Data (Deep Aquifer {aquifer})")
  plt.show()

  shap.summary_plot(shap_values_new.values, test_df[covariate_cols], show=False)
  plt.title(f"New/Test Data (Shallow Aquifer {aquifer})")
  plt.show()

In [ ]:
for aquifer in {'alluvial','non_alluvial'}:

  plt.figure(figsize=(8,4))
  sns.kdeplot(model.predict(train_df[covariate_cols]), label='Train Predictions')
  sns.kdeplot(model.predict(test_df[covariate_cols]), label='New Predictions')
  plt.title(f"{aquifer} Prediction Distribution")
  plt.legend()
  plt.show()

In [ ]:
for aquifer in {'alluvial','non_alluvial'}:
  df_combined = pd.concat([train_df[covariate_cols], test_df[covariate_cols]])
  labels = [0]*len(train_df) + [1]*len(test_df)

  from sklearn.ensemble import RandomForestClassifier
  from sklearn.model_selection import cross_val_score

  clf = RandomForestClassifier()
  auc = cross_val_score(clf, df_combined, labels, cv=5, scoring='roc_auc').mean()
  print(f"Covariate shift ROC-AUC: {auc:.3f}")

In [ ]:
from sklearn.preprocessing import StandardScaler

def run_normalized_shallow_deep_ols(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Normalize numeric features
            scaler = StandardScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()

            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()

In [ ]:
run_normalized_shallow_deep_ols(train_test_splits, numerical_cols, target_col)

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

def predict_shallow_with_deep_model(model_path, feature_cols, shallow_df, target_col, aquifer,season_label):
    """
    Load a model trained on deep aquifers and predict on shallow aquifer data.

    Parameters:
        model_path (str): Path to the saved model (e.g., .joblib file).
        feature_cols_path (str): Path to saved list of feature column names.
        shallow_df (pd.DataFrame): Shallow aquifer data (including target_col).
        target_col (str): The actual target variable.
        categorical_cols (list): List of categorical column names in shallow_df.

    Returns:
        pd.DataFrame: DataFrame with actual and predicted values.
    """
    # Load model and feature columns

    model = joblib.load(model_path)


    # One-hot encode categorical variables
    #shallow_df_encoded = pd.get_dummies(shallow_df.copy(), columns=categorical_cols, drop_first=True)

    # Align columns with training features
    shallow_df_encoded = shallow_df.reindex(columns=feature_cols, fill_value=0)

    # Extract input features and true values
    X_shallow = shallow_df_encoded
    y_shallow = shallow_df[target_col].values

    # Predict
    y_pred = model.predict(X_shallow)

    # Create result DataFrame
    result_df = pd.DataFrame({
        'Actual': y_shallow,
        'Predicted': y_pred
    })

    # Scatter Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(y_shallow, y_pred, alpha=0.6)
    plt.plot([min(y_shallow), max(y_shallow)], [min(y_shallow), max(y_shallow)], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{aquifer} Prediction on Shallow Aquifer using Deep Aquifer Model")
    plt.axis('equal')
    plt.grid(True)
    plt.show()

    return result_df


In [ ]:
for aquifer in {'alluvial','non_alluvial'}:
  if (aquifer=='alluvial'):
    test_df = shallow_alluvial
  else:
    test_df = shallow_non_alluvial
  model_path = os.path.join(model_dir, f"{aquifer}_{season_label}_ols_model.pkl")

  predict_shallow_with_deep_model(model_path,covariate_cols,test_df,target_col,aquifer,5)